# 051 - EmerG + Conditional Graph Diffusion

This notebook inherits the complete audited EmerG baseline from Notebook 04 and
adds a two-stage conditional denoising diffusion model over item-specific
feature-interaction graphs. It evaluates deterministic EmerG and uncertainty-aware
EmerG+GD on the same MovieLens-1M Cold/Warm A/Warm B/Warm C protocol.


## Notebook Linkage and Work Plan

**Inherited Stage A — Notebook 04:** verify the immutable `ml1m-coldstart-v1`
protocol, fit leakage-safe feature encoders, train/select/refit EmerG for the exact
five target seeds, evaluate Cold/Warm A/B/C, and publish verified baseline bundles.

**New Stage B — Graph Diffusion:**
1. Rebuild the selected tuning EmerG model without using validation labels for training.
2. Harvest converged item graphs from old items as pseudo-targets.
3. Train a shared conditional denoising model on continuous off-diagonal graph logits.
4. Sample multiple plausible graphs for each new item with a short DDIM-style chain.
5. Use few-shot CTR loss as sampling guidance for Warm A/B/C; the EmerG backbone stays frozen.
6. Refit the diffusion prior on final-train pseudo-graphs and evaluate the held-out evaluation items.
7. Export immutable EmerG+GD bundles and a paired EmerG versus EmerG+GD scenario table.

The comparison is paired by seed and scenario. Thresholds for EmerG+GD are selected
only from validation query labels and are frozen before evaluation.


In [1]:
from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import os
import platform
import random
import re
import shutil
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REQUIRED_PACKAGES = ["numpy", "pandas", "torch", "IPython"]
MISSING_PACKAGES = [
    package for package in REQUIRED_PACKAGES if importlib.util.find_spec(package) is None
]
if MISSING_PACKAGES:
    raise RuntimeError(
        "Notebook 051 inherited stage requires these packages in the active kernel: "
        + ", ".join(MISSING_PACKAGES)
        + ". Run it in the project ML/Kaggle environment used for model notebooks."
    )

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch import nn


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
PROTOCOL_RELATIVE_MANIFEST = Path("protocols/ml-1m/coldstart-v1/manifest.json")
MODEL_OUTPUT_ROOT = ARTIFACT_ROOT / "models" / "ml-1m" / "emerg-v1"
FAST_DEV_RUN = os.environ.get("COLDSTART_FAST_DEV_RUN", "0") == "1"

requested_device = os.environ.get("COLDSTART_DEVICE")
if requested_device:
    DEVICE = torch.device(requested_device)
    if DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("COLDSTART_DEVICE requests CUDA, but torch.cuda is unavailable")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGET_SEEDS = [2025, 7788, 9999, 3407, 4517]

base_epochs = int(os.environ.get("COLDSTART_EMERG_EPOCHS", "4"))
base_sample_rows = int(os.environ.get("COLDSTART_EMERG_SAMPLE_ROWS", "262144"))
base_warm_steps = int(os.environ.get("COLDSTART_EMERG_WARM_STEPS", "3"))
if FAST_DEV_RUN:
    base_epochs = min(base_epochs, 1)
    base_sample_rows = min(base_sample_rows, 32768)
    base_warm_steps = min(base_warm_steps, 1)

RUN_CONFIG_TEMPLATE: dict[str, Any] = {
    "schema_version": "emerg-baseline-v1",
    "fast_dev_run": FAST_DEV_RUN,
    "phase_order": ["Cold", "Warm A", "Warm B", "Warm C"],
    "field_order": [
        "user_id", "gender", "age", "occupation", "zip_code",
        "item_id", "release_year", "genres", "title",
    ],
    "item_graph_fields": ["item_id", "release_year", "genres", "title"],
    "max_title_tokens": 8,
    "max_genre_tokens": 6,
    "candidate_configs": [
        {
            "embedding_dim": 16,
            "hidden_dim": 64,
            "gnn_layers": 2,
            "learning_rate": 0.001,
            "warm_learning_rate": 0.01,
            "weight_decay": 1e-6,
            "epochs": base_epochs,
            "epoch_sample_rows": base_sample_rows,
            "batch_size": 4096,
            "warm_steps": base_warm_steps,
        },
        {
            "embedding_dim": 32,
            "hidden_dim": 96,
            "gnn_layers": 2,
            "learning_rate": 0.001,
            "warm_learning_rate": 0.01,
            "weight_decay": 1e-6,
            "epochs": base_epochs,
            "epoch_sample_rows": base_sample_rows,
            "batch_size": 4096,
            "warm_steps": base_warm_steps,
        },
    ][:1 if FAST_DEV_RUN else 2],
    "feature_policy": "fit categorical/text vocabularies on tuning-visible users/items; use PAD/UNK",
    "warmup_policy": "adapt local item embedding and graph delta sequentially: A, then B only, then C only",
}


def make_run_config(seed: int) -> dict[str, Any]:
    return {**RUN_CONFIG_TEMPLATE, "seed": int(seed)}


def run_config_hash(run_config: dict[str, Any]) -> str:
    return hashlib.sha256(
        json.dumps(
            run_config, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode()
    ).hexdigest()


def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def protocol_candidates() -> list[tuple[Path, Path]]:
    candidates: list[tuple[Path, Path]] = []
    explicit = os.environ.get("COLDSTART_PROTOCOL_ROOT")
    roots = [Path(explicit).expanduser()] if explicit else []
    roots.extend([PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT])

    for root in roots:
        pointer = root / PROTOCOL_RELATIVE_MANIFEST
        if pointer.is_file():
            candidates.append((root.resolve(), pointer.resolve()))
        direct = root / "manifest.json"
        if root.name == "coldstart-v1" and direct.is_file():
            candidates.append((root.parents[2].resolve(), direct.resolve()))

    if INPUT_ROOT.is_dir():
        for pointer in sorted(INPUT_ROOT.rglob("manifest.json")):
            parent = pointer.parent
            if (
                parent.name == "coldstart-v1"
                and parent.parent.name == "ml-1m"
                and parent.parent.parent.name == "protocols"
            ):
                candidates.append((pointer.parents[3].resolve(), pointer.resolve()))

    unique: list[tuple[Path, Path]] = []
    seen: set[str] = set()
    for root, pointer in candidates:
        key = str(pointer)
        if key not in seen:
            seen.add(key)
            unique.append((root, pointer))
    return unique


def load_verified_protocol(root: Path, pointer: Path) -> tuple[dict[str, Any], dict[str, pd.DataFrame]]:
    pointer_bytes = pointer.read_bytes()
    manifest = json.loads(pointer_bytes)
    if manifest.get("protocol_schema_version") != "ml1m-coldstart-v1":
        raise ValueError(f"Unexpected protocol schema: {manifest.get('protocol_schema_version')!r}")
    if manifest.get("protocol_status") != "PASS":
        raise ValueError(f"Protocol status is not PASS: {manifest.get('protocol_status')!r}")
    checks = manifest.get("checks")
    if (
        not isinstance(checks, list)
        or not checks
        or not all(isinstance(row, dict) and row.get("status") == "PASS" for row in checks)
    ):
        raise ValueError("One or more notebook-02 protocol checks did not pass")

    bundle_id = manifest.get("bundle_id")
    if (
        not isinstance(bundle_id, str)
        or not bundle_id
        or bundle_id in {".", ".."}
        or Path(bundle_id).name != bundle_id
    ):
        raise ValueError(f"Invalid protocol bundle id: {bundle_id!r}")
    bundle_manifest = resolve_inside(root, manifest["bundle_manifest"])
    expected_bundle_manifest = (
        pointer.parent / "generations" / bundle_id / "manifest.json"
    ).resolve()
    if bundle_manifest != expected_bundle_manifest:
        raise ValueError("Protocol bundle_manifest is not the immutable generation manifest")
    if bundle_manifest.read_bytes() != pointer_bytes:
        raise ValueError("Protocol pointer and immutable generation manifest differ")

    tables: dict[str, pd.DataFrame] = {}
    for name, artifact in manifest["artifacts"].items():
        path = resolve_inside(root, artifact["path"])
        if not path.is_file() or sha256_file(path) != artifact["sha256"]:
            raise ValueError(f"Protocol artifact verification failed: {name}")
        schema = manifest["output_schemas"][name]
        table = pd.read_csv(path, dtype=schema["read_csv_dtypes"])
        if list(table.columns) != schema["columns"] or len(table) != artifact["rows"]:
            raise ValueError(f"Protocol table contract failed: {name}")
        tables[name] = table
    return manifest, tables


PROTOCOL_ERRORS: list[str] = []
PROTOCOL_ROOT = None
PROTOCOL_POINTER = None
PROTOCOL_MANIFEST = None
TABLES = None
for candidate_root, candidate_pointer in protocol_candidates():
    try:
        PROTOCOL_MANIFEST, TABLES = load_verified_protocol(candidate_root, candidate_pointer)
        PROTOCOL_ROOT, PROTOCOL_POINTER = candidate_root, candidate_pointer
        break
    except Exception as error:
        PROTOCOL_ERRORS.append(f"{candidate_pointer}: {error}")

if PROTOCOL_MANIFEST is None or TABLES is None or PROTOCOL_POINTER is None:
    raise RuntimeError(
        "No valid notebook-02 protocol bundle found. Set COLDSTART_PROTOCOL_ROOT. "
        + " | ".join(PROTOCOL_ERRORS)
    )

TUNING_TRAIN = TABLES["tuning_train"]
FINAL_TRAIN = TABLES["final_train"]
VALIDATION_TASKS = TABLES["validation_tasks"]
EVALUATION_TASKS = TABLES["evaluation_tasks"]
USERS = TABLES["users"].sort_values("user_idx").reset_index(drop=True)
ITEMS = TABLES["items"].sort_values("item_idx").reset_index(drop=True)
N_USERS = int(USERS["user_idx"].max()) + 1
N_ITEMS = int(ITEMS["item_idx"].max()) + 1
PHASES = tuple(RUN_CONFIG_TEMPLATE["phase_order"])
PHASE_INCREMENT_ROLE = {"Warm A": "warm_a", "Warm B": "warm_b", "Warm C": "warm_c"}
PROTOCOL_POINTER_SHA256 = sha256_file(PROTOCOL_POINTER)

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "torch": torch.__version__,
            "device": str(DEVICE),
            "protocol_bundle": PROTOCOL_MANIFEST["bundle_id"],
            "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            "users": N_USERS,
            "items": N_ITEMS,
            "validation_query_rows": int(VALIDATION_TASKS["role"].eq("query").sum()),
            "evaluation_query_rows": int(EVALUATION_TASKS["role"].eq("query").sum()),
            "target_seeds": TARGET_SEEDS,
            "target_seed_count": len(TARGET_SEEDS),
        }
    ]
)

,execution_context,python,torch,device,protocol_bundle,protocol_pointer_sha256,users,items,validation_query_rows,evaluation_query_rows,target_seeds,target_seed_count
0,local,3.12.3,2.11.0+cu128,cuda,20260716T183148-e7b92de81a1e,5256c3aeda7f7e619fd60d9530397af6a207c7a83d80c4...,6040,2375,152762,57569,"[2025, 7788, 9999, 3407, 4517]",5


In [2]:
TOKEN_RE = re.compile(r"[a-z0-9]+")
PAD = "<PAD>"
UNK = "<UNK>"


def make_vocab(values: Iterable[Any]) -> dict[str, int]:
    vocab = {PAD: 0, UNK: 1}
    for value in sorted({str(item) for item in values if pd.notna(item)}):
        if value and value not in vocab:
            vocab[value] = len(vocab)
    return vocab


def encode_scalar(vocab: dict[str, int], value: Any) -> int:
    return vocab.get(str(value), vocab[UNK]) if pd.notna(value) else vocab[UNK]


def title_tokens(title: Any) -> list[str]:
    return TOKEN_RE.findall(str(title).lower())


def genre_tokens(genres: Any) -> list[str]:
    return [token for token in str(genres).split("|") if token]


def encode_sequence(vocab: dict[str, int], tokens: list[str], max_len: int) -> list[int]:
    encoded = [vocab.get(token, vocab[UNK]) for token in tokens[:max_len]]
    return encoded + [vocab[PAD]] * (max_len - len(encoded))


class FeatureStore:
    def __init__(
        self,
        users: pd.DataFrame,
        items: pd.DataFrame,
        train: pd.DataFrame,
        run_config: dict[str, Any],
    ):
        self.run_config = run_config
        train_user_ids = set(train["user_idx"].astype(int))
        train_item_ids = set(train["item_idx"].astype(int))
        train_users = users[users["user_idx"].isin(train_user_ids)]
        train_items = items[items["item_idx"].isin(train_item_ids)]

        self.gender_vocab = make_vocab(train_users["gender"])
        self.age_vocab = make_vocab(train_users["age"])
        self.occupation_vocab = make_vocab(train_users["occupation"])
        self.zip_vocab = make_vocab(train_users["zip_code"])
        self.genre_vocab = make_vocab(
            token for value in train_items["genres"] for token in genre_tokens(value)
        )
        self.title_vocab = make_vocab(
            token for value in train_items["title"] for token in title_tokens(value)
        )

        train_years = pd.to_numeric(train_items["release_year"], errors="coerce").astype("float32")
        self.release_mean = float(train_years.mean())
        self.release_std = float(train_years.std() if train_years.std() > 0 else 1.0)

        users = users.sort_values("user_idx").reset_index(drop=True)
        items = items.sort_values("item_idx").reset_index(drop=True)
        self.user_gender = np.array([encode_scalar(self.gender_vocab, x) for x in users["gender"]], dtype=np.int64)
        self.user_age = np.array([encode_scalar(self.age_vocab, x) for x in users["age"]], dtype=np.int64)
        self.user_occupation = np.array([encode_scalar(self.occupation_vocab, x) for x in users["occupation"]], dtype=np.int64)
        self.user_zip = np.array([encode_scalar(self.zip_vocab, x) for x in users["zip_code"]], dtype=np.int64)

        years = pd.to_numeric(items["release_year"], errors="coerce").astype("float32")
        years = years.fillna(self.release_mean)
        self.item_release = ((years.to_numpy(dtype=np.float32) - self.release_mean) / self.release_std).astype(np.float32)
        self.item_genres = np.array(
            [
                encode_sequence(
                    self.genre_vocab, genre_tokens(value), int(self.run_config["max_genre_tokens"])
                )
                for value in items["genres"]
            ],
            dtype=np.int64,
        )
        self.item_titles = np.array(
            [
                encode_sequence(
                    self.title_vocab, title_tokens(value), int(self.run_config["max_title_tokens"])
                )
                for value in items["title"]
            ],
            dtype=np.int64,
        )
        self.field_sizes = {
            "user_id": N_USERS,
            "gender": len(self.gender_vocab),
            "age": len(self.age_vocab),
            "occupation": len(self.occupation_vocab),
            "zip_code": len(self.zip_vocab),
            "item_id": N_ITEMS,
            "genres": len(self.genre_vocab),
            "title": len(self.title_vocab),
        }

    def arrays(self, table: pd.DataFrame) -> dict[str, np.ndarray]:
        return {
            "source_row": table["source_row"].to_numpy(dtype=np.int64),
            "user_id": table["user_id"].to_numpy(dtype=np.int64),
            "user_idx": table["user_idx"].to_numpy(dtype=np.int64),
            "item_id": table["item_id"].to_numpy(dtype=np.int64),
            "item_idx": table["item_idx"].to_numpy(dtype=np.int64),
            "label": table["label"].to_numpy(dtype=np.float32),
        }

    def batch(self, arrays: dict[str, np.ndarray], rows: np.ndarray) -> tuple[dict[str, torch.Tensor], torch.Tensor]:
        user_idx = arrays["user_idx"][rows]
        item_idx = arrays["item_idx"][rows]
        features = {
            "user_id": torch.as_tensor(user_idx, dtype=torch.long, device=DEVICE),
            "gender": torch.as_tensor(self.user_gender[user_idx], dtype=torch.long, device=DEVICE),
            "age": torch.as_tensor(self.user_age[user_idx], dtype=torch.long, device=DEVICE),
            "occupation": torch.as_tensor(self.user_occupation[user_idx], dtype=torch.long, device=DEVICE),
            "zip_code": torch.as_tensor(self.user_zip[user_idx], dtype=torch.long, device=DEVICE),
            "item_id": torch.as_tensor(item_idx, dtype=torch.long, device=DEVICE),
            "release_year": torch.as_tensor(self.item_release[item_idx], dtype=torch.float32, device=DEVICE),
            "genres": torch.as_tensor(self.item_genres[item_idx], dtype=torch.long, device=DEVICE),
            "title": torch.as_tensor(self.item_titles[item_idx], dtype=torch.long, device=DEVICE),
        }
        labels = torch.as_tensor(arrays["label"][rows], dtype=torch.float32, device=DEVICE)
        return features, labels

    def contract(self) -> dict[str, Any]:
        vocabs = {
            "gender": self.gender_vocab,
            "age": self.age_vocab,
            "occupation": self.occupation_vocab,
            "zip_code": self.zip_vocab,
            "genres": self.genre_vocab,
            "title": self.title_vocab,
        }
        return {
            "schema_version": "emerg-feature-contract-v1",
            "field_order": self.run_config["field_order"],
            "item_graph_fields": self.run_config["item_graph_fields"],
            "max_title_tokens": self.run_config["max_title_tokens"],
            "max_genre_tokens": self.run_config["max_genre_tokens"],
            "release_year_mean": self.release_mean,
            "release_year_std": self.release_std,
            "field_sizes": self.field_sizes,
            "vocab_sha256": {
                name: sha256_bytes(
                    json.dumps(vocab, sort_keys=True, allow_nan=False).encode()
                )
                for name, vocab in vocabs.items()
            },
            "vocab_sizes": {name: len(vocab) for name, vocab in vocabs.items()},
            "vocabs": vocabs,
        }

In [3]:
FIELD_ORDER = list(RUN_CONFIG_TEMPLATE["field_order"])
ITEM_GRAPH_FIELDS = list(RUN_CONFIG_TEMPLATE["item_graph_fields"])
ITEM_GRAPH_POSITIONS = [FIELD_ORDER.index(name) for name in ITEM_GRAPH_FIELDS]


class EmerGCTR(nn.Module):
    def __init__(self, feature_sizes: dict[str, int], embedding_dim: int, hidden_dim: int, gnn_layers: int):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_fields = len(FIELD_ORDER)
        self.gnn_layers = gnn_layers
        self.embeddings = nn.ModuleDict(
            {
                "user_id": nn.Embedding(feature_sizes["user_id"], embedding_dim),
                "gender": nn.Embedding(feature_sizes["gender"], embedding_dim, padding_idx=0),
                "age": nn.Embedding(feature_sizes["age"], embedding_dim, padding_idx=0),
                "occupation": nn.Embedding(feature_sizes["occupation"], embedding_dim, padding_idx=0),
                "zip_code": nn.Embedding(feature_sizes["zip_code"], embedding_dim, padding_idx=0),
                "item_id": nn.Embedding(feature_sizes["item_id"], embedding_dim),
                "genres": nn.Embedding(feature_sizes["genres"], embedding_dim, padding_idx=0),
                "title": nn.Embedding(feature_sizes["title"], embedding_dim, padding_idx=0),
            }
        )
        self.release_weight = nn.Parameter(torch.empty(embedding_dim))
        self.item_graph_delta = nn.Embedding(feature_sizes["item_id"], self.num_fields * self.num_fields)
        self.graph_generator = nn.Sequential(
            nn.Linear(len(ITEM_GRAPH_FIELDS) * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.num_fields * self.num_fields),
        )
        self.graph_layers = nn.ModuleList(
            [nn.Linear(embedding_dim, embedding_dim, bias=False) for _ in range(gnn_layers)]
        )
        self.head = nn.Sequential(
            nn.Linear(self.num_fields * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.register_buffer("identity_graph", torch.eye(self.num_fields))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)
                if module.padding_idx is not None:
                    with torch.no_grad():
                        module.weight[module.padding_idx].zero_()
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
        nn.init.normal_(self.release_weight, std=0.02)
        nn.init.zeros_(self.item_graph_delta.weight)

    def sequence_embedding(self, name: str, tokens: torch.Tensor) -> torch.Tensor:
        embedded = self.embeddings[name](tokens)
        mask = tokens.ne(0).float().unsqueeze(-1)
        denom = mask.sum(dim=1).clamp_min(1.0)
        return (embedded * mask).sum(dim=1) / denom

    def field_embeddings(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
    ) -> torch.Tensor:
        item_embedding = self.embeddings["item_id"](features["item_id"])
        if local_item_embedding is not None:
            item_embedding = local_item_embedding.unsqueeze(0).expand_as(item_embedding)
        fields = [
            self.embeddings["user_id"](features["user_id"]),
            self.embeddings["gender"](features["gender"]),
            self.embeddings["age"](features["age"]),
            self.embeddings["occupation"](features["occupation"]),
            self.embeddings["zip_code"](features["zip_code"]),
            item_embedding,
            features["release_year"].unsqueeze(1) * self.release_weight.unsqueeze(0),
            self.sequence_embedding("genres", features["genres"]),
            self.sequence_embedding("title", features["title"]),
        ]
        return torch.stack(fields, dim=1)

    def generated_graph(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
        local_graph_delta: torch.Tensor | None = None,
    ) -> torch.Tensor:
        field_emb = self.field_embeddings(features, local_item_embedding)
        item_context = field_emb[:, ITEM_GRAPH_POSITIONS, :].reshape(field_emb.shape[0], -1)
        raw_graph = self.graph_generator(item_context)
        if local_graph_delta is None:
            raw_graph = raw_graph + self.item_graph_delta(features["item_id"])
        else:
            raw_graph = raw_graph + local_graph_delta.unsqueeze(0).expand_as(raw_graph)
        graph = torch.sigmoid(raw_graph).reshape(-1, self.num_fields, self.num_fields)
        graph = 0.5 * (graph + graph.transpose(1, 2))
        eye = self.identity_graph.unsqueeze(0).to(graph.device)
        return graph * (1.0 - eye) + eye

    def forward(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
        local_graph_delta: torch.Tensor | None = None,
    ) -> torch.Tensor:
        h = self.field_embeddings(features, local_item_embedding)
        graph = self.generated_graph(features, local_item_embedding, local_graph_delta)
        for layer in self.graph_layers:
            message = torch.bmm(graph, layer(h))
            h = h + F.relu(message)
        return self.head(h.reshape(h.shape[0], -1)).squeeze(1)


def roc_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    negatives = int(len(labels) - positives)
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    ranks = np.arange(1, len(scores) + 1, dtype=np.float64)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        if end - start > 1:
            ranks[start:end] = ranks[start:end].mean()
        start = end
    original_ranks = np.empty_like(ranks)
    original_ranks[order] = ranks
    return float((original_ranks[labels == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives))


def best_f1_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    labels = labels.astype(np.int64)
    order = np.argsort(-scores, kind="mergesort")
    sorted_labels = labels[order]
    sorted_scores = scores[order]
    tp = np.cumsum(sorted_labels)
    fp = np.cumsum(1 - sorted_labels)
    fn = int(sorted_labels.sum()) - tp
    denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denominator, out=np.zeros_like(tp, dtype=np.float64), where=denominator > 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_scores[1:] != sorted_scores[:-1], True]
    )
    best = int(group_ends[np.argmax(f1[group_ends])])
    return float(sorted_scores[best]), float(f1[best])


def binary_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    labels = labels.astype(np.int64)
    predictions = scores >= threshold
    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    tn = int(((predictions == 0) & (labels == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "rows": int(len(labels)),
        "positives": int(labels.sum()),
        "threshold": float(threshold),
        "accuracy": float((tp + tn) / len(labels)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": roc_auc(labels, scores),
        "predicted_positive_rate": float(predictions.mean()),
        "score_mean": float(scores.mean()),
        "score_std": float(scores.std()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


def summarize_predictions(
    predictions: pd.DataFrame,
    split: str,
    thresholds: dict[str, float] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    threshold_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    for phase in PHASES:
        phase_predictions = predictions[predictions["phase"].eq(phase)]
        labels = phase_predictions["label"].to_numpy(dtype=np.int64)
        scores = phase_predictions["score"].to_numpy(dtype=np.float64)
        if thresholds is None:
            threshold, best_f1 = best_f1_threshold(labels, scores)
        else:
            threshold, best_f1 = float(thresholds[phase]), np.nan
        threshold_rows.append(
            {"split": split, "phase": phase, "threshold": threshold, "validation_best_f1": best_f1}
        )
        metric_rows.append({"split": split, "phase": phase, **binary_metrics(labels, scores, threshold)})
    return pd.DataFrame(threshold_rows), pd.DataFrame(metric_rows)


def numeric_tables_are_finite(*tables: pd.DataFrame) -> bool:
    for table in tables:
        numeric = table.select_dtypes(include=[np.number])
        if numeric.empty or not np.isfinite(numeric.to_numpy(dtype=np.float64)).all():
            return False
    return True

In [4]:
def gradient_flow_check(
    feature_store: FeatureStore,
    run_config: dict[str, Any],
    seed: int,
) -> dict[str, Any]:
    set_random_seed(seed)
    config = dict(run_config["candidate_configs"][0])
    model = EmerGCTR(
        feature_store.field_sizes,
        embedding_dim=int(config["embedding_dim"]),
        hidden_dim=int(config["hidden_dim"]),
        gnn_layers=int(config["gnn_layers"]),
    ).to(DEVICE)
    arrays = feature_store.arrays(TUNING_TRAIN.iloc[: min(256, len(TUNING_TRAIN))])
    rows = np.arange(len(arrays["label"]), dtype=np.int64)
    features, labels = feature_store.batch(arrays, rows)
    loss = F.binary_cross_entropy_with_logits(model(features), labels)
    model.zero_grad(set_to_none=True)
    loss.backward()
    graph_grad_norm = 0.0
    for parameter in model.graph_generator.parameters():
        if parameter.grad is not None:
            graph_grad_norm += float(parameter.grad.detach().abs().sum().cpu())
    result = {
        "loss": float(loss.detach().cpu()),
        "graph_generator_grad_norm": graph_grad_norm,
        "status": "PASS" if graph_grad_norm > 0 else "FAIL",
    }
    del model
    return result

In [5]:
def train_emerg_model(
    feature_store: FeatureStore,
    train_table: pd.DataFrame,
    config: dict[str, Any],
    seed: int,
    run_name: str,
) -> tuple[EmerGCTR, pd.DataFrame, float]:
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    set_random_seed(seed)
    arrays = feature_store.arrays(train_table)
    model = EmerGCTR(
        feature_store.field_sizes,
        embedding_dim=int(config["embedding_dim"]),
        hidden_dim=int(config["hidden_dim"]),
        gnn_layers=int(config["gnn_layers"]),
    ).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=float(config["learning_rate"]), weight_decay=float(config["weight_decay"])
    )
    history: list[dict[str, Any]] = []
    n_rows = len(arrays["label"])
    sample_rows = min(int(config["epoch_sample_rows"]), n_rows)
    batch_size = int(config["batch_size"])

    for epoch in range(1, int(config["epochs"]) + 1):
        model.train()
        if sample_rows < n_rows:
            order = rng.choice(n_rows, size=sample_rows, replace=False)
        else:
            order = rng.permutation(n_rows)
        total_loss = 0.0
        total_examples = 0
        for start_idx in range(0, len(order), batch_size):
            rows = order[start_idx : start_idx + batch_size]
            features, labels = feature_store.batch(arrays, rows)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach().cpu()) * len(rows)
            total_examples += len(rows)
        history.append(
            {
                "run_name": run_name,
                "epoch": epoch,
                "loss": total_loss / max(total_examples, 1),
                "examples": total_examples,
            }
        )
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model, pd.DataFrame(history), time.perf_counter() - start


def freeze_parameters(model: nn.Module, value: bool) -> list[bool]:
    previous = [parameter.requires_grad for parameter in model.parameters()]
    for parameter in model.parameters():
        parameter.requires_grad_(value)
    return previous


def restore_requires_grad(model: nn.Module, previous: list[bool]) -> None:
    for parameter, requires_grad in zip(model.parameters(), previous):
        parameter.requires_grad_(requires_grad)


def predict_table(
    feature_store: FeatureStore,
    model: EmerGCTR,
    table: pd.DataFrame,
    local_item_embedding: torch.Tensor | None = None,
    local_graph_delta: torch.Tensor | None = None,
    batch_size: int = 8192,
) -> np.ndarray:
    arrays = feature_store.arrays(table)
    scores: list[np.ndarray] = []
    model.eval()
    with torch.no_grad():
        for start_idx in range(0, len(table), batch_size):
            rows = np.arange(start_idx, min(start_idx + batch_size, len(table)), dtype=np.int64)
            features, _ = feature_store.batch(arrays, rows)
            logits = model(features, local_item_embedding, local_graph_delta)
            scores.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(scores).astype(np.float32)


def adapt_local_state(
    feature_store: FeatureStore,
    model: EmerGCTR,
    support_table: pd.DataFrame,
    local_item_embedding: torch.Tensor,
    local_graph_delta: torch.Tensor,
    config: dict[str, Any],
) -> tuple[torch.Tensor, torch.Tensor, float]:
    if support_table.empty:
        return local_item_embedding, local_graph_delta, 0.0
    arrays = feature_store.arrays(support_table.reset_index(drop=True))
    rows = np.arange(len(support_table), dtype=np.int64)
    optimizer = torch.optim.Adam([local_item_embedding, local_graph_delta], lr=float(config["warm_learning_rate"]))
    last_loss = 0.0
    model.eval()
    for _ in range(int(config["warm_steps"])):
        features, labels = feature_store.batch(arrays, rows)
        optimizer.zero_grad(set_to_none=True)
        logits = model(features, local_item_embedding, local_graph_delta)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
    return local_item_embedding.detach().requires_grad_(), local_graph_delta.detach().requires_grad_(), last_loss


def graph_stats(
    feature_store: FeatureStore,
    model: EmerGCTR,
    one_row: pd.DataFrame,
    phase: str,
    local_item_embedding: torch.Tensor | None,
    local_graph_delta: torch.Tensor | None,
    local_loss: float,
) -> dict[str, Any]:
    arrays = feature_store.arrays(one_row.reset_index(drop=True))
    features, _ = feature_store.batch(arrays, np.array([0], dtype=np.int64))
    with torch.no_grad():
        graph = model.generated_graph(features, local_item_embedding, local_graph_delta)[0].detach().cpu().numpy()
    return {
        "item_id": int(one_row["item_id"].iloc[0]),
        "item_idx": int(one_row["item_idx"].iloc[0]),
        "phase": phase,
        "graph_mean": float(graph.mean()),
        "graph_std": float(graph.std()),
        "graph_density_gt_0_5": float((graph > 0.5).mean()),
        "graph_diag_mean": float(np.diag(graph).mean()),
        "local_delta_norm": float(local_graph_delta.detach().norm().cpu()) if local_graph_delta is not None else 0.0,
        "last_warm_loss": float(local_loss),
    }


def score_tasks_with_warmup(
    feature_store: FeatureStore,
    model: EmerGCTR,
    tasks: pd.DataFrame,
    config: dict[str, Any],
    split: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    predictions: list[pd.DataFrame] = []
    diagnostics: list[dict[str, Any]] = []
    previous_requires_grad = freeze_parameters(model, False)
    try:
        sorted_tasks = tasks.sort_values(["item_idx", "item_rank", "source_row"]).reset_index(drop=True)
        for item_idx, item_rows in sorted_tasks.groupby("item_idx", sort=True):
            item_rows = item_rows.reset_index(drop=True)
            query_rows = item_rows[item_rows["role"].eq("query")].reset_index(drop=True)
            base_embedding = model.embeddings["item_id"].weight[int(item_idx)].detach().clone().requires_grad_()
            base_delta = model.item_graph_delta.weight[int(item_idx)].detach().clone().requires_grad_()

            cold_scores = predict_table(feature_store, model, query_rows)
            cold_predictions = query_rows[
                ["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]
            ].copy()
            cold_predictions.insert(0, "phase", "Cold")
            cold_predictions["score"] = cold_scores
            predictions.append(cold_predictions)
            diagnostics.append(
                graph_stats(feature_store, model, query_rows.iloc[[0]], "Cold", None, None, 0.0)
            )

            local_embedding = base_embedding
            local_delta = base_delta
            for phase, role in PHASE_INCREMENT_ROLE.items():
                support_rows = item_rows[item_rows["role"].eq(role)].reset_index(drop=True)
                local_embedding, local_delta, warm_loss = adapt_local_state(
                    feature_store, model, support_rows, local_embedding, local_delta, config
                )
                phase_scores = predict_table(
                    feature_store, model, query_rows, local_embedding, local_delta
                )
                phase_predictions = query_rows[
                    ["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]
                ].copy()
                phase_predictions.insert(0, "phase", phase)
                phase_predictions["score"] = phase_scores
                predictions.append(phase_predictions)
                diagnostics.append(
                    graph_stats(
                        feature_store,
                        model,
                        query_rows.iloc[[0]],
                        phase,
                        local_embedding,
                        local_delta,
                        warm_loss,
                    )
                )
    finally:
        restore_requires_grad(model, previous_requires_grad)
    prediction_table = pd.concat(predictions, ignore_index=True)
    diagnostics_table = pd.DataFrame(diagnostics)
    diagnostics_table.insert(0, "split", split)
    return prediction_table, diagnostics_table

In [6]:
def tune_seed(feature_store: FeatureStore, run_config: dict[str, Any]) -> dict[str, Any]:
    training_history_parts: list[pd.DataFrame] = []
    config_summary_rows: list[dict[str, Any]] = []
    selected_payload: dict[str, Any] | None = None
    seed = int(run_config["seed"])

    for config_index, config in enumerate(run_config["candidate_configs"], start=1):
        config_name = f"cfg{config_index:02d}_G{config['gnn_layers']}_D{config['embedding_dim']}"
        model, history, training_seconds = train_emerg_model(
            feature_store,
            TUNING_TRAIN,
            config,
            seed=seed + config_index,
            run_name=config_name,
        )
        history["config_name"] = config_name
        training_history_parts.append(history)

        try:
            validation_predictions, validation_diagnostics = score_tasks_with_warmup(
                feature_store, model, VALIDATION_TASKS, config, split="validation"
            )
        finally:
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        thresholds, metrics = summarize_predictions(validation_predictions, split="validation")
        thresholds["config_name"] = config_name
        metrics["config_name"] = config_name
        validation_diagnostics["config_name"] = config_name

        mean_f1 = float(metrics["f1"].mean())
        mean_auc = float(metrics["roc_auc"].mean())
        summary = {
            "config_name": config_name,
            "mean_validation_f1": mean_f1,
            "mean_validation_auc": mean_auc,
            "training_seconds": float(training_seconds),
            **config,
        }
        config_summary_rows.append(summary)
        candidate_payload = {
            "config_name": config_name,
            "config": dict(config),
            "summary": summary,
            "thresholds": thresholds,
            "metrics": metrics,
            "predictions": validation_predictions,
            "diagnostics": validation_diagnostics,
        }
        if selected_payload is None or (mean_f1, mean_auc) > (
            selected_payload["summary"]["mean_validation_f1"],
            selected_payload["summary"]["mean_validation_auc"],
        ):
            selected_payload = candidate_payload

    if selected_payload is None:
        raise RuntimeError(f"No EmerG candidate was trained for seed {seed}")

    validation_thresholds = selected_payload["thresholds"].copy()
    return {
        "training_history": pd.concat(training_history_parts, ignore_index=True),
        "config_summary": pd.DataFrame(config_summary_rows).sort_values(
            ["mean_validation_f1", "mean_validation_auc"], ascending=False
        ),
        "selected_config_name": str(selected_payload["config_name"]),
        "selected_config": dict(selected_payload["config"]),
        "selected_summary": dict(selected_payload["summary"]),
        "validation_predictions": selected_payload["predictions"].copy(),
        "validation_thresholds": validation_thresholds,
        "validation_metrics": selected_payload["metrics"].copy(),
        "validation_diagnostics": selected_payload["diagnostics"].copy(),
        "threshold_by_phase": dict(
            zip(validation_thresholds["phase"], validation_thresholds["threshold"])
        ),
    }

In [7]:
def pre_export_check(
    checks: list[dict[str, Any]],
    name: str,
    condition: bool,
    observed: Any,
    expected: Any,
) -> None:
    checks.append(
        {"check": name, "status": "PASS" if condition else "FAIL", "observed": observed, "expected": expected}
    )


def train_evaluate_seed(
    feature_store: FeatureStore,
    run_config: dict[str, Any],
    tuning: dict[str, Any],
    gradient_check: dict[str, Any],
) -> dict[str, Any]:
    seed = int(run_config["seed"])
    selected_config_name = tuning["selected_config_name"]
    selected_config = tuning["selected_config"]
    final_model, final_history, final_training_seconds = train_emerg_model(
        feature_store,
        FINAL_TRAIN,
        selected_config,
        seed=seed + 10_000,
        run_name=f"final_refit_{selected_config_name}",
    )
    final_history["config_name"] = selected_config_name
    training_history = pd.concat(
        [tuning["training_history"], final_history], ignore_index=True
    )

    try:
        evaluation_predictions, evaluation_diagnostics = score_tasks_with_warmup(
            feature_store,
            final_model,
            EVALUATION_TASKS,
            selected_config,
            split="evaluation",
        )
        _, evaluation_metrics = summarize_predictions(
            evaluation_predictions,
            split="evaluation",
            thresholds=tuning["threshold_by_phase"],
        )
    except Exception:
        raise

    evaluation_metrics["config_name"] = selected_config_name
    graph_diagnostics = pd.concat(
        [tuning["validation_diagnostics"], evaluation_diagnostics], ignore_index=True
    )
    graph_diagnostics["selected_config_name"] = selected_config_name
    graph_diagnostics["final_training_seconds"] = float(final_training_seconds)
    graph_diagnostics["device"] = str(DEVICE)

    checks: list[dict[str, Any]] = []
    expected_validation_predictions = int(
        VALIDATION_TASKS["role"].eq("query").sum() * len(PHASES)
    )
    expected_evaluation_predictions = int(
        EVALUATION_TASKS["role"].eq("query").sum() * len(PHASES)
    )
    final_excludes_evaluation_items = set(FINAL_TRAIN["item_idx"]).isdisjoint(
        set(EVALUATION_TASKS["item_idx"])
    )
    feature_contract_has_pad_unk = all(
        vocab.get(PAD) == 0 and vocab.get(UNK) == 1
        for vocab in [
            feature_store.gender_vocab,
            feature_store.age_vocab,
            feature_store.occupation_vocab,
            feature_store.zip_vocab,
            feature_store.genre_vocab,
            feature_store.title_vocab,
        ]
    )
    training_values_finite = numeric_tables_are_finite(training_history)
    validation_values_finite = numeric_tables_are_finite(
        tuning["config_summary"],
        tuning["validation_thresholds"],
        tuning["validation_metrics"],
        tuning["validation_predictions"],
    )
    evaluation_values_finite = numeric_tables_are_finite(
        evaluation_predictions, evaluation_metrics
    )

    pre_export_check(
        checks,
        "protocol schema",
        PROTOCOL_MANIFEST["protocol_schema_version"] == "ml1m-coldstart-v1",
        PROTOCOL_MANIFEST["protocol_schema_version"],
        "ml1m-coldstart-v1",
    )
    pre_export_check(
        checks, "gradient flow", gradient_check["status"] == "PASS", gradient_check["status"], "PASS"
    )
    pre_export_check(
        checks,
        "phase thresholds complete",
        set(tuning["threshold_by_phase"]) == set(PHASES),
        sorted(tuning["threshold_by_phase"]),
        sorted(PHASES),
    )
    pre_export_check(
        checks,
        "validation predictions complete",
        len(tuning["validation_predictions"]) == expected_validation_predictions,
        len(tuning["validation_predictions"]),
        expected_validation_predictions,
    )
    pre_export_check(
        checks,
        "evaluation predictions complete",
        len(evaluation_predictions) == expected_evaluation_predictions,
        len(evaluation_predictions),
        expected_evaluation_predictions,
    )
    pre_export_check(
        checks,
        "evaluation phases complete",
        set(evaluation_metrics["phase"]) == set(PHASES),
        sorted(evaluation_metrics["phase"]),
        sorted(PHASES),
    )
    pre_export_check(
        checks,
        "final train excludes evaluation items",
        final_excludes_evaluation_items,
        final_excludes_evaluation_items,
        True,
    )
    pre_export_check(
        checks,
        "feature contract has PAD/UNK",
        feature_contract_has_pad_unk,
        feature_contract_has_pad_unk,
        True,
    )
    pre_export_check(
        checks,
        "graph diagnostics complete",
        set(graph_diagnostics["phase"]) == set(PHASES),
        sorted(graph_diagnostics["phase"].unique()),
        sorted(PHASES),
    )
    pre_export_check(
        checks,
        "training values finite",
        training_values_finite,
        training_values_finite,
        True,
    )
    pre_export_check(
        checks,
        "validation values finite",
        validation_values_finite,
        validation_values_finite,
        True,
    )
    pre_export_check(
        checks,
        "evaluation values finite",
        evaluation_values_finite,
        evaluation_values_finite,
        True,
    )

    failed_checks = [row["check"] for row in checks if row["status"] != "PASS"]
    if failed_checks:
        raise RuntimeError(
            f"EmerG semantic checks failed for seed {seed}: {', '.join(failed_checks)}"
        )

    return {
        "final_model": final_model,
        "final_training_seconds": float(final_training_seconds),
        "selected_config_name": selected_config_name,
        "selected_config": selected_config,
        "selected_summary": tuning["selected_summary"],
        "threshold_by_phase": tuning["threshold_by_phase"],
        "checks": checks,
        "output_tables": {
            "training_history": training_history,
            "config_summary": tuning["config_summary"],
            "validation_thresholds": tuning["validation_thresholds"],
            "validation_metrics": tuning["validation_metrics"],
            "validation_predictions": tuning["validation_predictions"],
            "evaluation_metrics": evaluation_metrics,
            "evaluation_predictions": evaluation_predictions,
            "graph_diagnostics": graph_diagnostics,
        },
    }


def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {column: dtype_name(dtype) for column, dtype in table.dtypes.items()},
    }


def verify_generation(manifest: dict[str, Any], expected_seed: int) -> dict[str, Any]:
    verification = {
        "manifest_verified": False,
        "artifact_hashes_verified": False,
        "error": "",
    }
    try:
        generation_root = MODEL_OUTPUT_ROOT / "generations" / manifest["bundle_id"]
        manifest_path = resolve_inside(ARTIFACT_ROOT, manifest["bundle_manifest"])
        manifest_text = manifest_path.read_text(encoding="utf-8")
        loaded_manifest = json.loads(manifest_text)
        expected_manifest_path = (generation_root / "manifest.json").resolve()
        expected_artifacts = {
            "training_history",
            "config_summary",
            "validation_thresholds",
            "validation_metrics",
            "validation_predictions",
            "evaluation_metrics",
            "evaluation_predictions",
            "graph_diagnostics",
            "selected_config",
            "feature_contract",
            "final_model_checkpoint",
        }
        manifest_contract_ok = (
            manifest_path == expected_manifest_path
            and manifest_text
            == json.dumps(
                json_ready(manifest), indent=2, sort_keys=True, allow_nan=False
            )
            + "\n"
            and loaded_manifest.get("model_status") == "PASS"
            and loaded_manifest.get("run_config", {}).get("seed") == int(expected_seed)
            and loaded_manifest.get("run_config_sha256")
            == run_config_hash(loaded_manifest["run_config"])
            and loaded_manifest.get("bundle_manifest")
            == relative_output(expected_manifest_path)
            and set(loaded_manifest.get("artifacts", {})) == expected_artifacts
        )
        artifact_paths_inside_generation = True
        artifact_hashes_verified = True
        for artifact in loaded_manifest.get("artifacts", {}).values():
            artifact_path = resolve_inside(ARTIFACT_ROOT, artifact["path"])
            try:
                artifact_path.resolve().relative_to(generation_root.resolve())
            except ValueError:
                artifact_paths_inside_generation = False
            if not artifact_path.is_file() or sha256_file(artifact_path) != artifact["sha256"]:
                artifact_hashes_verified = False

        verification["manifest_verified"] = bool(
            manifest_contract_ok
            and artifact_paths_inside_generation
            and "final_model_checkpoint" in loaded_manifest.get("artifacts", {})
            and "feature_contract" in loaded_manifest.get("artifacts", {})
        )
        verification["artifact_hashes_verified"] = bool(artifact_hashes_verified)
    except Exception as error:
        verification["error"] = f"{type(error).__name__}: {error}"
    verification["verified"] = bool(
        verification["manifest_verified"] and verification["artifact_hashes_verified"]
    )
    return verification


def export_seed(
    feature_contract: dict[str, Any],
    run_config: dict[str, Any],
    run_payload: dict[str, Any],
    bundle_id: str,
) -> dict[str, Any]:
    seed = int(run_config["seed"])
    config_sha256 = run_config_hash(run_config)
    staging_root = MODEL_OUTPUT_ROOT / f".staging-{bundle_id}"
    generation_root = MODEL_OUTPUT_ROOT / "generations" / bundle_id
    pointer_path = MODEL_OUTPUT_ROOT / "manifest.json"
    output_tables = run_payload["output_tables"]
    output_artifacts: dict[str, Any] = {}
    staging_root.mkdir(parents=True, exist_ok=False)
    previous_pointer_text = (
        pointer_path.read_text(encoding="utf-8") if pointer_path.is_file() else None
    )
    generation_published = False
    pointer_write_attempted = False

    selected_config_payload = {
        "schema_version": run_config["schema_version"],
        "selected_config_name": run_payload["selected_config_name"],
        "selected_config": run_payload["selected_config"],
        "selected_summary": run_payload["selected_summary"],
        "threshold_by_phase": run_payload["threshold_by_phase"],
        "run_config_sha256": config_sha256,
        "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
    }

    try:
        for name, table in output_tables.items():
            staging_path = staging_root / f"{name}.csv"
            published_path = generation_root / f"{name}.csv"
            write_csv(staging_path, table)
            output_artifacts[name] = {
                "path": relative_output(published_path),
                "sha256": sha256_file(staging_path),
                "rows": len(table),
            }

        selected_config_path = staging_root / "selected_config.json"
        write_text(
            selected_config_path,
            json.dumps(
                json_ready(selected_config_payload),
                indent=2,
                sort_keys=True,
                allow_nan=False,
            )
            + "\n",
        )
        output_artifacts["selected_config"] = {
            "path": relative_output(generation_root / "selected_config.json"),
            "sha256": sha256_file(selected_config_path),
        }

        feature_contract_path = staging_root / "feature_contract.json"
        write_text(
            feature_contract_path,
            json.dumps(
                json_ready(feature_contract),
                indent=2,
                sort_keys=True,
                allow_nan=False,
            )
            + "\n",
        )
        output_artifacts["feature_contract"] = {
            "path": relative_output(generation_root / "feature_contract.json"),
            "sha256": sha256_file(feature_contract_path),
        }

        checkpoint_path = staging_root / "final_model.pt"
        torch.save(
            {
                "schema_version": run_config["schema_version"],
                "model_state_dict": run_payload["final_model"].state_dict(),
                "selected_config": run_payload["selected_config"],
                "feature_contract": feature_contract,
                "threshold_by_phase": run_payload["threshold_by_phase"],
                "n_users": N_USERS,
                "n_items": N_ITEMS,
                "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            checkpoint_path,
        )
        output_artifacts["final_model_checkpoint"] = {
            "path": relative_output(generation_root / "final_model.pt"),
            "sha256": sha256_file(checkpoint_path),
        }

        evaluation_metrics = output_tables["evaluation_metrics"]
        evaluation_predictions = output_tables["evaluation_predictions"]
        manifest = {
            "model_schema_version": run_config["schema_version"],
            "model_status": "PASS",
            "model_name": "EmerG",
            "bundle_id": bundle_id,
            "bundle_manifest": relative_output(generation_root / "manifest.json"),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "upstream_protocol": {
                "schema_version": PROTOCOL_MANIFEST["protocol_schema_version"],
                "bundle_id": PROTOCOL_MANIFEST["bundle_id"],
                "pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            "run_config": run_config,
            "run_config_sha256": config_sha256,
            "feature_contract_sha256": sha256_file(feature_contract_path),
            "selected_config": selected_config_payload,
            "checks": run_payload["checks"],
            "summary": {
                "selected_config_name": run_payload["selected_config_name"],
                "mean_validation_f1": run_payload["selected_summary"]["mean_validation_f1"],
                "mean_validation_auc": run_payload["selected_summary"]["mean_validation_auc"],
                "mean_evaluation_f1": float(evaluation_metrics["f1"].mean()),
                "mean_evaluation_auc": float(evaluation_metrics["roc_auc"].mean()),
                "evaluation_prediction_rows": len(evaluation_predictions),
                "final_training_seconds": run_payload["final_training_seconds"],
            },
            "artifacts": output_artifacts,
            "output_schemas": {
                name: csv_schema(table) for name, table in output_tables.items()
            },
            "phase_contract": PROTOCOL_MANIFEST["phase_contract"],
            "training_contract": {
                "tuning_base": "tuning_train from notebook 02",
                "final_refit_base": "final_train from notebook 02 after thresholds/config are frozen",
                "features": "audited user/item side information only; no interaction_count/cohort/timestamp/rank features",
                "warmup": run_config["warmup_policy"],
                "thresholds": "selected on validation query rows only and frozen for evaluation",
            },
        }

        manifest_text = (
            json.dumps(
                json_ready(manifest), indent=2, sort_keys=True, allow_nan=False
            )
            + "\n"
        )
        write_text(staging_root / "manifest.json", manifest_text)
        generation_root.parent.mkdir(parents=True, exist_ok=True)
        staging_root.replace(generation_root)
        generation_published = True

        verification = verify_generation(manifest, seed)
        if not verification["verified"]:
            raise RuntimeError(
                f"Published EmerG generation failed verification for seed {seed}: "
                + (verification["error"] or str(verification))
            )
        pointer_write_attempted = True
        write_text(pointer_path, manifest_text)
        pointer_verified = pointer_path.read_bytes() == (generation_root / "manifest.json").read_bytes()
        if not pointer_verified:
            raise RuntimeError(f"EmerG pointer verification failed for seed {seed}")
    except Exception:
        try:
            if pointer_write_attempted:
                if previous_pointer_text is None:
                    pointer_path.unlink(missing_ok=True)
                else:
                    write_text(pointer_path, previous_pointer_text)
        finally:
            if staging_root.exists():
                shutil.rmtree(staging_root, ignore_errors=True)
            if generation_published and generation_root.exists():
                shutil.rmtree(generation_root)
        raise

    return {
        "manifest": manifest,
        "generation_manifest": generation_root / "manifest.json",
        "pointer": pointer_path,
        "verification": verification,
    }

In [8]:
MULTI_SEED_RUN_REGISTRY: list[dict[str, Any]] = []
COMPLETED_EXPORTS: list[dict[str, Any]] = []

for target_seed in TARGET_SEEDS:
    seed_started = time.perf_counter()
    planned_bundle_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        + f"-s{target_seed}-"
        + uuid.uuid4().hex[:12]
    )
    registry_row = {
        "seed": target_seed,
        "status": "BLOCKED",
        "bundle_id": "",
        "generation_published": False,
        "manifest_verified": False,
        "artifact_hashes_verified": False,
        "mean_evaluation_f1": np.nan,
        "error": "",
    }
    feature_store = None
    tuning = None
    run_payload = None
    try:
        run_config = make_run_config(target_seed)
        set_random_seed(target_seed)
        feature_store = FeatureStore(USERS, ITEMS, TUNING_TRAIN, run_config)
        feature_contract = feature_store.contract()
        gradient_check = gradient_flow_check(feature_store, run_config, target_seed)
        if gradient_check["status"] != "PASS":
            raise RuntimeError("EmerG graph generator is not on the autograd path")

        tuning = tune_seed(feature_store, run_config)
        run_payload = train_evaluate_seed(
            feature_store, run_config, tuning, gradient_check
        )
        export = export_seed(
            feature_contract, run_config, run_payload, planned_bundle_id
        )
        manifest = export["manifest"]
        verification = export["verification"]
        registry_row.update(
            {
                "status": "READY",
                "bundle_id": manifest["bundle_id"],
                "generation_published": True,
                "manifest_verified": verification["manifest_verified"],
                "artifact_hashes_verified": verification["artifact_hashes_verified"],
                "mean_evaluation_f1": manifest["summary"]["mean_evaluation_f1"],
            }
        )
        COMPLETED_EXPORTS.append(
            {
                "seed": target_seed,
                "manifest": manifest,
                "generation_manifest": export["generation_manifest"],
                "pointer": export["pointer"],
                "registry_row": registry_row,
            }
        )
    except Exception as error:
        generation_manifest = (
            MODEL_OUTPUT_ROOT / "generations" / planned_bundle_id / "manifest.json"
        )
        if generation_manifest.is_file():
            registry_row["bundle_id"] = planned_bundle_id
            registry_row["generation_published"] = True
        registry_row["error"] = f"{type(error).__name__}: {error}"
    finally:
        registry_row["elapsed_seconds"] = time.perf_counter() - seed_started
        MULTI_SEED_RUN_REGISTRY.append(registry_row)
        run_payload = None
        tuning = None
        feature_store = None
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

for completed in COMPLETED_EXPORTS:
    verification = verify_generation(completed["manifest"], completed["seed"])
    registry_row = completed["registry_row"]
    registry_row["manifest_verified"] = verification["manifest_verified"]
    registry_row["artifact_hashes_verified"] = verification["artifact_hashes_verified"]
    if not verification["verified"]:
        registry_row["status"] = "BLOCKED"
        registry_row["error"] = verification["error"] or "Final generation verification failed"

completed_seeds = [completed["seed"] for completed in COMPLETED_EXPORTS]
bundle_ids = [completed["manifest"]["bundle_id"] for completed in COMPLETED_EXPORTS]
exact_targets_attempted = [row["seed"] for row in MULTI_SEED_RUN_REGISTRY] == TARGET_SEEDS
exact_targets_exported = completed_seeds == TARGET_SEEDS
unique_generations = len(bundle_ids) == len(set(bundle_ids)) == len(TARGET_SEEDS)
pointer_matches_last_completed = True
if COMPLETED_EXPORTS:
    last_completed = COMPLETED_EXPORTS[-1]
    try:
        pointer_matches_last_completed = (
            last_completed["pointer"].is_file()
            and last_completed["pointer"].read_bytes()
            == last_completed["generation_manifest"].read_bytes()
        )
    except Exception as error:
        pointer_matches_last_completed = False
        last_completed["registry_row"]["error"] = f"{type(error).__name__}: {error}"
    if not pointer_matches_last_completed:
        last_completed["registry_row"]["status"] = "BLOCKED"
        if not last_completed["registry_row"]["error"]:
            last_completed["registry_row"]["error"] = (
                "Mutable pointer does not match the last completed generation"
            )

all_seed_rows_ready = all(row["status"] == "READY" for row in MULTI_SEED_RUN_REGISTRY)
all_generations_verified = all(
    row["manifest_verified"] and row["artifact_hashes_verified"]
    for row in MULTI_SEED_RUN_REGISTRY
)

EMERG_PASS = all(
    [
        exact_targets_attempted,
        exact_targets_exported,
        unique_generations,
        all_seed_rows_ready,
        all_generations_verified,
        pointer_matches_last_completed,
    ]
)
RUN_REGISTRY = pd.DataFrame(MULTI_SEED_RUN_REGISTRY)
display(
    RUN_REGISTRY[
        [
            "seed",
            "status",
            "bundle_id",
            "generation_published",
            "manifest_verified",
            "artifact_hashes_verified",
            "mean_evaluation_f1",
            "elapsed_seconds",
            "error",
        ]
    ]
)
display(Markdown("### Notebook 051 Stage A — inherited EmerG baseline: " + ("READY" if EMERG_PASS else "BLOCKED")))

if not EMERG_PASS:
    raise RuntimeError(
        "EmerG multi-seed run is BLOCKED; every exact target seed must export a verified immutable generation"
    )

display(
    Markdown(
        "**Next notebook:** implement the proposed DGD model in notebook 05 using the same notebook-02 protocol. "
        "Keep LightGCN and EmerG manifests as baseline outputs for notebook 08 comparisons."
    )
)

,seed,status,bundle_id,generation_published,manifest_verified,artifact_hashes_verified,mean_evaluation_f1,elapsed_seconds,error
0,2025,READY,20260717T024905-s2025-bf2814aee7cc,True,True,True,0.668103,70.489223,
1,7788,READY,20260717T025015-s7788-50f1feb72787,True,True,True,0.669826,68.171274,
2,9999,READY,20260717T025124-s9999-c791b1bbf9bc,True,True,True,0.669661,69.060444,
3,3407,READY,20260717T025233-s3407-1d55e5872364,True,True,True,0.671497,66.479409,
4,4517,READY,20260717T025339-s4517-40be8703c620,True,True,True,0.667121,36.373250,


### Notebook 051 Stage A — inherited EmerG baseline: READY

**Next notebook:** implement the proposed DGD model in notebook 05 using the same notebook-02 protocol. Keep LightGCN and EmerG manifests as baseline outputs for notebook 08 comparisons.

## Stage B — Conditional Graph Diffusion over EmerG graphs

### Method implemented here

This notebook uses the **two-stage pseudo-graph strategy** because the protocol has no
ground-truth feature graphs:

1. Train the inherited EmerG backbone on old items.
2. Harvest each old item's converged continuous adjacency matrix.
3. Vectorize the upper-triangular off-diagonal probabilities, transform them to logits,
   and train a conditional DDPM noise predictor.
4. Condition the denoiser on side-information embeddings (`release_year`, `genres`, and
   `title`) so a truly unseen item does not depend on a trained item-ID embedding.
5. Use a short DDIM-style reverse chain to sample several graphs and ensemble CTR scores.
6. For Warm A/B/C, differentiate the observed support CTR loss with respect to the current
   clean-graph estimate and move the sample in the loss-reducing direction. This keeps the
   graph close to the learned diffusion prior rather than directly optimizing a free adjacency.

The graph has only nine feature nodes, so its 36 undirected off-diagonal edges are small
enough for multi-sample guided inference. Hard top-K is intentionally not used; sigmoid
edge probabilities remain differentiable throughout guidance.

### Related foundations

- Jo, Lee, and Hwang, *Score-based Generative Modeling of Graphs via the System of SDEs* (GDSS).
- Vignac et al., *DiGress: Discrete Denoising Diffusion for Graph Generation*.
- Song, Meng, and Ermon, *Denoising Diffusion Implicit Models* (DDIM).

This is a compact continuous-edge implementation tailored to EmerG's tiny weighted graphs,
not a line-for-line reproduction of GDSS or DiGress.


In [9]:
# Graph-diffusion configuration and modules.
GD_MODEL_OUTPUT_ROOT = ARTIFACT_ROOT / "models" / "ml-1m" / "emerg-gd-v1"
GD_CONDITION_FIELDS = ["release_year", "genres", "title"]
GD_CONDITION_POSITIONS = [FIELD_ORDER.index(name) for name in GD_CONDITION_FIELDS]
GRAPH_NODE_COUNT = len(FIELD_ORDER)
GRAPH_EDGE_INDEX = torch.triu_indices(GRAPH_NODE_COUNT, GRAPH_NODE_COUNT, offset=1)
GRAPH_EDGE_COUNT = int(GRAPH_EDGE_INDEX.shape[1])

base_gd_epochs = int(os.environ.get("COLDSTART_GD_EPOCHS", "24"))
base_gd_batch_size = int(os.environ.get("COLDSTART_GD_BATCH_SIZE", "256"))
base_gd_samples = int(os.environ.get("COLDSTART_GD_SAMPLES", "4"))
base_gd_sampler_steps = int(os.environ.get("COLDSTART_GD_SAMPLER_STEPS", "6"))
base_gd_timesteps = int(os.environ.get("COLDSTART_GD_TIMESTEPS", "64"))
base_gd_guidance = float(os.environ.get("COLDSTART_GD_GUIDANCE_SCALE", "0.20"))
if FAST_DEV_RUN:
    base_gd_epochs = min(base_gd_epochs, 2)
    base_gd_batch_size = min(base_gd_batch_size, 128)
    base_gd_samples = min(base_gd_samples, 2)
    base_gd_sampler_steps = min(base_gd_sampler_steps, 3)
    base_gd_timesteps = min(base_gd_timesteps, 16)

GD_CONFIG_TEMPLATE: dict[str, Any] = {
    "schema_version": "emerg-graph-diffusion-v1",
    "strategy": "two-stage conditional continuous-edge DDPM with DDIM-style guided sampling",
    "condition_fields": GD_CONDITION_FIELDS,
    "diffusion_timesteps": base_gd_timesteps,
    "sampler_steps": base_gd_sampler_steps,
    "samples_per_item": base_gd_samples,
    "denoiser_hidden_dim": 192,
    "time_embedding_dim": 32,
    "denoiser_epochs": base_gd_epochs,
    "denoiser_batch_size": base_gd_batch_size,
    "denoiser_learning_rate": 1e-3,
    "denoiser_weight_decay": 1e-6,
    "reconstruction_weight": 0.10,
    "guidance_scale": base_gd_guidance,
    "clip_normalized_x0": 6.0,
    "edge_probability_epsilon": 1e-4,
    "warm_support_policy": "cumulative support: A; A+B; A+B+C",
    "prediction_policy": "mean probability across independently initialized graph samples",
}


def make_gd_config(seed: int, selected_emerg_config: dict[str, Any]) -> dict[str, Any]:
    return {
        **GD_CONFIG_TEMPLATE,
        "seed": int(seed),
        "selected_emerg_config": dict(selected_emerg_config),
    }


def timestep_embedding(timesteps: torch.Tensor, dimension: int) -> torch.Tensor:
    half = dimension // 2
    if half == 0:
        return timesteps.float().unsqueeze(1)
    denominator = max(half - 1, 1)
    frequencies = torch.exp(
        -np.log(10_000.0)
        * torch.arange(half, device=timesteps.device, dtype=torch.float32)
        / denominator
    )
    angles = timesteps.float().unsqueeze(1) * frequencies.unsqueeze(0)
    embedding = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
    if dimension % 2:
        embedding = F.pad(embedding, (0, 1))
    return embedding


class ConditionalGraphDenoiser(nn.Module):
    def __init__(
        self,
        edge_dim: int,
        condition_dim: int,
        hidden_dim: int,
        time_dim: int,
    ):
        super().__init__()
        self.time_dim = int(time_dim)
        self.input_projection = nn.Linear(edge_dim, hidden_dim)
        self.condition_projection = nn.Linear(condition_dim, hidden_dim)
        self.time_projection = nn.Sequential(
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.blocks = nn.ModuleList(
            [
                nn.Sequential(
                    nn.LayerNorm(hidden_dim),
                    nn.Linear(hidden_dim, hidden_dim * 2),
                    nn.SiLU(),
                    nn.Linear(hidden_dim * 2, hidden_dim),
                )
                for _ in range(3)
            ]
        )
        self.output_projection = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, edge_dim),
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(
        self,
        noisy_edges: torch.Tensor,
        timesteps: torch.Tensor,
        condition: torch.Tensor,
    ) -> torch.Tensor:
        hidden = (
            self.input_projection(noisy_edges)
            + self.condition_projection(condition)
            + self.time_projection(timestep_embedding(timesteps, self.time_dim))
        )
        for block in self.blocks:
            hidden = hidden + block(hidden)
        return self.output_projection(hidden)


class ConditionalGraphDiffusion(nn.Module):
    def __init__(
        self,
        edge_dim: int,
        condition_dim: int,
        hidden_dim: int,
        time_dim: int,
        timesteps: int,
    ):
        super().__init__()
        self.edge_dim = int(edge_dim)
        self.timesteps = int(timesteps)
        self.denoiser = ConditionalGraphDenoiser(
            edge_dim=edge_dim,
            condition_dim=condition_dim,
            hidden_dim=hidden_dim,
            time_dim=time_dim,
        )
        betas = torch.linspace(1e-4, 2e-2, self.timesteps, dtype=torch.float32)
        alphas = 1.0 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alpha_bars", alpha_bars)
        self.register_buffer("sqrt_alpha_bars", torch.sqrt(alpha_bars))
        self.register_buffer("sqrt_one_minus_alpha_bars", torch.sqrt(1.0 - alpha_bars))
        self.register_buffer("edge_mean", torch.zeros(edge_dim))
        self.register_buffer("edge_std", torch.ones(edge_dim))

    def set_edge_normalizer(self, mean: torch.Tensor, std: torch.Tensor) -> None:
        with torch.no_grad():
            self.edge_mean.copy_(mean.to(self.edge_mean))
            self.edge_std.copy_(std.clamp_min(1e-3).to(self.edge_std))

    def normalize(self, edge_logits: torch.Tensor) -> torch.Tensor:
        return (edge_logits - self.edge_mean) / self.edge_std

    def denormalize(self, normalized_edges: torch.Tensor) -> torch.Tensor:
        return normalized_edges * self.edge_std + self.edge_mean

    def q_sample(
        self,
        clean_edges: torch.Tensor,
        timesteps: torch.Tensor,
        noise: torch.Tensor,
    ) -> torch.Tensor:
        signal = self.sqrt_alpha_bars[timesteps].unsqueeze(1)
        noise_scale = self.sqrt_one_minus_alpha_bars[timesteps].unsqueeze(1)
        return signal * clean_edges + noise_scale * noise

    def predict_clean(
        self,
        noisy_edges: torch.Tensor,
        timesteps: torch.Tensor,
        predicted_noise: torch.Tensor,
    ) -> torch.Tensor:
        signal = self.sqrt_alpha_bars[timesteps].unsqueeze(1).clamp_min(1e-6)
        noise_scale = self.sqrt_one_minus_alpha_bars[timesteps].unsqueeze(1)
        return (noisy_edges - noise_scale * predicted_noise) / signal


In [10]:
def load_torch_checkpoint(path: Path) -> dict[str, Any]:
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def baseline_artifact_path(manifest: dict[str, Any], artifact_name: str) -> Path:
    return resolve_inside(ARTIFACT_ROOT, manifest["artifacts"][artifact_name]["path"])


def load_baseline_model(
    feature_store: FeatureStore,
    baseline_manifest: dict[str, Any],
) -> tuple[EmerGCTR, dict[str, Any]]:
    selected_config = dict(baseline_manifest["selected_config"]["selected_config"])
    model = EmerGCTR(
        feature_store.field_sizes,
        embedding_dim=int(selected_config["embedding_dim"]),
        hidden_dim=int(selected_config["hidden_dim"]),
        gnn_layers=int(selected_config["gnn_layers"]),
    ).to(DEVICE)
    checkpoint = load_torch_checkpoint(
        baseline_artifact_path(baseline_manifest, "final_model_checkpoint")
    )
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    return model, selected_config


def representative_item_rows(train_table: pd.DataFrame) -> pd.DataFrame:
    columns = ["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]
    return (
        train_table.sort_values(["item_idx", "source_row"])
        .drop_duplicates("item_idx", keep="first")[columns]
        .reset_index(drop=True)
    )


def harvest_pseudo_graph_dataset(
    feature_store: FeatureStore,
    emerg_model: EmerGCTR,
    train_table: pd.DataFrame,
    gd_config: dict[str, Any],
    batch_size: int = 1024,
) -> dict[str, Any]:
    rows_table = representative_item_rows(train_table)
    arrays = feature_store.arrays(rows_table)
    condition_parts: list[torch.Tensor] = []
    edge_parts: list[torch.Tensor] = []
    epsilon = float(gd_config["edge_probability_epsilon"])
    emerg_model.eval()
    with torch.no_grad():
        for start_idx in range(0, len(rows_table), batch_size):
            rows = np.arange(
                start_idx,
                min(start_idx + batch_size, len(rows_table)),
                dtype=np.int64,
            )
            features, _ = feature_store.batch(arrays, rows)
            field_embeddings = emerg_model.field_embeddings(features)
            condition = field_embeddings[:, GD_CONDITION_POSITIONS, :].reshape(len(rows), -1)
            graph = emerg_model.generated_graph(features)
            edge_probabilities = graph[
                :, GRAPH_EDGE_INDEX[0].to(graph.device), GRAPH_EDGE_INDEX[1].to(graph.device)
            ]
            edge_logits = torch.logit(edge_probabilities.clamp(epsilon, 1.0 - epsilon))
            condition_parts.append(condition.detach().cpu())
            edge_parts.append(edge_logits.detach().cpu())

    conditions = torch.cat(condition_parts, dim=0)
    edge_logits = torch.cat(edge_parts, dim=0)
    pseudo_stats = pd.DataFrame(
        {
            "item_idx": rows_table["item_idx"].astype(np.int64),
            "item_id": rows_table["item_id"].astype(np.int64),
            "edge_logit_mean": edge_logits.mean(dim=1).numpy(),
            "edge_logit_std": edge_logits.std(dim=1, unbiased=False).numpy(),
            "edge_probability_mean": torch.sigmoid(edge_logits).mean(dim=1).numpy(),
            "edge_probability_std": torch.sigmoid(edge_logits).std(dim=1, unbiased=False).numpy(),
        }
    )
    return {
        "conditions": conditions,
        "edge_logits": edge_logits,
        "rows": rows_table,
        "stats": pseudo_stats,
    }


def train_conditional_graph_diffusion(
    pseudo_dataset: dict[str, Any],
    gd_config: dict[str, Any],
    seed: int,
    run_name: str,
) -> tuple[ConditionalGraphDiffusion, pd.DataFrame, float]:
    start = time.perf_counter()
    set_random_seed(seed)
    conditions = pseudo_dataset["conditions"].float()
    edge_logits = pseudo_dataset["edge_logits"].float()
    edge_mean = edge_logits.mean(dim=0)
    edge_std = edge_logits.std(dim=0, unbiased=False).clamp_min(1e-3)
    diffusion = ConditionalGraphDiffusion(
        edge_dim=GRAPH_EDGE_COUNT,
        condition_dim=int(conditions.shape[1]),
        hidden_dim=int(gd_config["denoiser_hidden_dim"]),
        time_dim=int(gd_config["time_embedding_dim"]),
        timesteps=int(gd_config["diffusion_timesteps"]),
    ).to(DEVICE)
    diffusion.set_edge_normalizer(edge_mean.to(DEVICE), edge_std.to(DEVICE))
    optimizer = torch.optim.AdamW(
        diffusion.parameters(),
        lr=float(gd_config["denoiser_learning_rate"]),
        weight_decay=float(gd_config["denoiser_weight_decay"]),
    )
    generator = torch.Generator().manual_seed(seed)
    row_count = len(edge_logits)
    batch_size = min(int(gd_config["denoiser_batch_size"]), row_count)
    history: list[dict[str, Any]] = []

    for epoch in range(1, int(gd_config["denoiser_epochs"]) + 1):
        diffusion.train()
        order = torch.randperm(row_count, generator=generator)
        total_loss = 0.0
        total_noise_loss = 0.0
        total_reconstruction_loss = 0.0
        total_examples = 0
        for start_idx in range(0, row_count, batch_size):
            batch_rows = order[start_idx : start_idx + batch_size]
            clean_logits = edge_logits[batch_rows].to(DEVICE)
            condition = conditions[batch_rows].to(DEVICE)
            clean = diffusion.normalize(clean_logits)
            timesteps = torch.randint(
                0,
                diffusion.timesteps,
                (len(batch_rows),),
                device=DEVICE,
            )
            noise = torch.randn_like(clean)
            noisy = diffusion.q_sample(clean, timesteps, noise)
            predicted_noise = diffusion.denoiser(noisy, timesteps, condition)
            noise_loss = F.mse_loss(predicted_noise, noise)
            predicted_clean = diffusion.predict_clean(noisy, timesteps, predicted_noise)
            reconstruction_loss = F.mse_loss(predicted_clean, clean)
            loss = noise_loss + float(gd_config["reconstruction_weight"]) * reconstruction_loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(diffusion.parameters(), max_norm=5.0)
            optimizer.step()

            examples = len(batch_rows)
            total_examples += examples
            total_loss += float(loss.detach().cpu()) * examples
            total_noise_loss += float(noise_loss.detach().cpu()) * examples
            total_reconstruction_loss += float(reconstruction_loss.detach().cpu()) * examples

        history.append(
            {
                "run_name": run_name,
                "epoch": epoch,
                "loss": total_loss / max(total_examples, 1),
                "noise_loss": total_noise_loss / max(total_examples, 1),
                "reconstruction_loss": total_reconstruction_loss / max(total_examples, 1),
                "examples": total_examples,
            }
        )
    diffusion.eval()
    return diffusion, pd.DataFrame(history), time.perf_counter() - start


def graph_diffusion_gradient_check(
    feature_store: FeatureStore,
    emerg_model: EmerGCTR,
    diffusion: ConditionalGraphDiffusion,
    train_table: pd.DataFrame,
) -> dict[str, Any]:
    sample = train_table.head(min(64, len(train_table))).reset_index(drop=True)
    pseudo = harvest_pseudo_graph_dataset(feature_store, emerg_model, sample, GD_CONFIG_TEMPLATE)
    clean_logits = pseudo["edge_logits"].to(DEVICE)
    condition = pseudo["conditions"].to(DEVICE)
    clean = diffusion.normalize(clean_logits)
    timesteps = torch.zeros(len(clean), dtype=torch.long, device=DEVICE)
    noise = torch.randn_like(clean)
    noisy = diffusion.q_sample(clean, timesteps, noise)
    diffusion.zero_grad(set_to_none=True)
    prediction = diffusion.denoiser(noisy, timesteps, condition)
    loss = F.mse_loss(prediction, noise)
    loss.backward()
    gradient_norm = float(
        sum(
            parameter.grad.detach().norm().cpu()
            for parameter in diffusion.denoiser.parameters()
            if parameter.grad is not None
        )
    )
    diffusion.zero_grad(set_to_none=True)
    return {
        "status": "PASS" if np.isfinite(gradient_norm) and gradient_norm > 0 else "FAIL",
        "gradient_norm": gradient_norm,
        "loss": float(loss.detach().cpu()),
    }


In [11]:
def edge_logits_to_graphs(edge_logits: torch.Tensor) -> torch.Tensor:
    probabilities = torch.sigmoid(edge_logits)
    graphs = torch.zeros(
        (len(edge_logits), GRAPH_NODE_COUNT, GRAPH_NODE_COUNT),
        dtype=edge_logits.dtype,
        device=edge_logits.device,
    )
    row_index = GRAPH_EDGE_INDEX[0].to(edge_logits.device)
    column_index = GRAPH_EDGE_INDEX[1].to(edge_logits.device)
    graphs[:, row_index, column_index] = probabilities
    graphs[:, column_index, row_index] = probabilities
    diagonal = torch.arange(GRAPH_NODE_COUNT, device=edge_logits.device)
    graphs[:, diagonal, diagonal] = 1.0
    return graphs


def emerg_forward_with_graph(
    emerg_model: EmerGCTR,
    features: dict[str, torch.Tensor],
    graph: torch.Tensor,
) -> torch.Tensor:
    hidden = emerg_model.field_embeddings(features)
    if graph.ndim == 2:
        graph = graph.unsqueeze(0).expand(hidden.shape[0], -1, -1)
    elif graph.shape[0] == 1 and hidden.shape[0] > 1:
        graph = graph.expand(hidden.shape[0], -1, -1)
    if graph.shape[0] != hidden.shape[0]:
        raise ValueError(
            f"Graph batch {graph.shape[0]} does not match feature batch {hidden.shape[0]}"
        )
    for layer in emerg_model.graph_layers:
        message = torch.bmm(graph, layer(hidden))
        hidden = hidden + F.relu(message)
    return emerg_model.head(hidden.reshape(hidden.shape[0], -1)).squeeze(1)


def condition_for_item(
    feature_store: FeatureStore,
    emerg_model: EmerGCTR,
    one_row: pd.DataFrame,
) -> torch.Tensor:
    arrays = feature_store.arrays(one_row.iloc[[0]].reset_index(drop=True))
    features, _ = feature_store.batch(arrays, np.array([0], dtype=np.int64))
    with torch.no_grad():
        field_embeddings = emerg_model.field_embeddings(features)
        return field_embeddings[:, GD_CONDITION_POSITIONS, :].reshape(1, -1)


def reverse_timestep_schedule(total_steps: int, sampler_steps: int) -> list[int]:
    values = np.linspace(total_steps - 1, 0, num=min(total_steps, sampler_steps), dtype=np.int64)
    return list(dict.fromkeys(int(value) for value in values))


def sample_guided_graphs(
    feature_store: FeatureStore,
    emerg_model: EmerGCTR,
    diffusion: ConditionalGraphDiffusion,
    item_condition: torch.Tensor,
    gd_config: dict[str, Any],
    sample_seed: int,
    support_table: pd.DataFrame | None = None,
) -> tuple[torch.Tensor, dict[str, float]]:
    sample_count = int(gd_config["samples_per_item"])
    generator_device = DEVICE if DEVICE.type == "cuda" else torch.device("cpu")
    generator = torch.Generator(device=generator_device).manual_seed(int(sample_seed))
    noisy = torch.randn(
        (sample_count, GRAPH_EDGE_COUNT),
        generator=generator,
        device=DEVICE,
    )
    condition = item_condition.to(DEVICE).expand(sample_count, -1)
    schedule = reverse_timestep_schedule(
        diffusion.timesteps,
        int(gd_config["sampler_steps"]),
    )
    guidance_scale = float(gd_config["guidance_scale"])
    guidance_losses: list[float] = []
    support_features = None
    support_labels = None
    if support_table is not None and not support_table.empty:
        support_arrays = feature_store.arrays(support_table.reset_index(drop=True))
        support_rows = np.arange(len(support_table), dtype=np.int64)
        support_features, support_labels = feature_store.batch(support_arrays, support_rows)

    diffusion.eval()
    emerg_model.eval()
    for schedule_index, timestep in enumerate(schedule):
        timestep_batch = torch.full(
            (sample_count,), timestep, dtype=torch.long, device=DEVICE
        )
        with torch.no_grad():
            predicted_noise = diffusion.denoiser(noisy, timestep_batch, condition)
            clean = diffusion.predict_clean(noisy, timestep_batch, predicted_noise)

        if support_features is not None and support_labels is not None and guidance_scale > 0:
            guided_clean = clean.detach().requires_grad_(True)
            guided_graphs = edge_logits_to_graphs(diffusion.denormalize(guided_clean))
            support_losses = []
            for sample_index in range(sample_count):
                logits = emerg_forward_with_graph(
                    emerg_model,
                    support_features,
                    guided_graphs[sample_index],
                )
                support_losses.append(
                    F.binary_cross_entropy_with_logits(logits, support_labels)
                )
            guidance_loss = torch.stack(support_losses).sum()
            gradient = torch.autograd.grad(guidance_loss, guided_clean, only_inputs=True)[0]
            gradient = gradient / gradient.norm(dim=1, keepdim=True).clamp_min(1e-6)
            time_weight = float(diffusion.sqrt_one_minus_alpha_bars[timestep].detach().cpu())
            clean = (
                guided_clean
                - guidance_scale * max(time_weight, 0.05) * gradient
            ).detach()
            guidance_losses.append(float(guidance_loss.detach().cpu()) / sample_count)
            signal = diffusion.sqrt_alpha_bars[timestep].clamp_min(1e-6)
            noise_scale = diffusion.sqrt_one_minus_alpha_bars[timestep].clamp_min(1e-6)
            predicted_noise = (noisy - signal * clean) / noise_scale

        clean = clean.clamp(
            -float(gd_config["clip_normalized_x0"]),
            float(gd_config["clip_normalized_x0"]),
        )
        next_timestep = schedule[schedule_index + 1] if schedule_index + 1 < len(schedule) else -1
        if next_timestep >= 0:
            noisy = (
                diffusion.sqrt_alpha_bars[next_timestep] * clean
                + diffusion.sqrt_one_minus_alpha_bars[next_timestep] * predicted_noise
            )
        else:
            noisy = clean

    edge_logits = diffusion.denormalize(noisy)
    graphs = edge_logits_to_graphs(edge_logits).detach()
    return graphs, {
        "guidance_loss_mean": float(np.mean(guidance_losses)) if guidance_losses else 0.0,
        "guidance_loss_last": float(guidance_losses[-1]) if guidance_losses else 0.0,
    }


def predict_graph_ensemble(
    feature_store: FeatureStore,
    emerg_model: EmerGCTR,
    query_table: pd.DataFrame,
    graphs: torch.Tensor,
    batch_size: int = 8192,
) -> tuple[np.ndarray, np.ndarray]:
    arrays = feature_store.arrays(query_table.reset_index(drop=True))
    per_graph_scores: list[np.ndarray] = []
    emerg_model.eval()
    with torch.no_grad():
        for graph in graphs:
            graph_scores: list[np.ndarray] = []
            for start_idx in range(0, len(query_table), batch_size):
                rows = np.arange(
                    start_idx,
                    min(start_idx + batch_size, len(query_table)),
                    dtype=np.int64,
                )
                features, _ = feature_store.batch(arrays, rows)
                logits = emerg_forward_with_graph(emerg_model, features, graph)
                graph_scores.append(torch.sigmoid(logits).detach().cpu().numpy())
            per_graph_scores.append(np.concatenate(graph_scores))
    score_matrix = np.stack(per_graph_scores, axis=0)
    return (
        score_matrix.mean(axis=0).astype(np.float32),
        score_matrix.std(axis=0).astype(np.float32),
    )


def graph_ensemble_stats(
    graphs: torch.Tensor,
    split: str,
    phase: str,
    item_rows: pd.DataFrame,
    guidance_stats: dict[str, float],
    sampling_seconds: float,
) -> dict[str, Any]:
    graph_array = graphs.detach().cpu().numpy()
    return {
        "split": split,
        "item_id": int(item_rows["item_id"].iloc[0]),
        "item_idx": int(item_rows["item_idx"].iloc[0]),
        "phase": phase,
        "graph_sample_count": int(len(graphs)),
        "graph_mean": float(graph_array.mean()),
        "graph_std_within": float(graph_array.std()),
        "graph_between_sample_std": float(graph_array.std(axis=0).mean()),
        "graph_density_gt_0_5": float((graph_array > 0.5).mean()),
        "graph_diag_mean": float(np.diagonal(graph_array, axis1=1, axis2=2).mean()),
        "guidance_loss_mean": guidance_stats["guidance_loss_mean"],
        "guidance_loss_last": guidance_stats["guidance_loss_last"],
        "sampling_seconds": float(sampling_seconds),
    }


def score_tasks_with_graph_diffusion(
    feature_store: FeatureStore,
    emerg_model: EmerGCTR,
    diffusion: ConditionalGraphDiffusion,
    tasks: pd.DataFrame,
    gd_config: dict[str, Any],
    split: str,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    predictions: list[pd.DataFrame] = []
    diagnostics: list[dict[str, Any]] = []
    previous_emerg = freeze_parameters(emerg_model, False)
    previous_diffusion = freeze_parameters(diffusion, False)
    phase_seed_offset = {"Cold": 0, "Warm A": 1, "Warm B": 2, "Warm C": 3}
    try:
        sorted_tasks = tasks.sort_values(["item_idx", "item_rank", "source_row"]).reset_index(drop=True)
        for item_idx, item_rows in sorted_tasks.groupby("item_idx", sort=True):
            item_rows = item_rows.reset_index(drop=True)
            query_rows = item_rows[item_rows["role"].eq("query")].reset_index(drop=True)
            item_condition = condition_for_item(feature_store, emerg_model, query_rows)
            cumulative_support: list[pd.DataFrame] = []

            for phase in PHASES:
                if phase != "Cold":
                    role = PHASE_INCREMENT_ROLE[phase]
                    cumulative_support.append(
                        item_rows[item_rows["role"].eq(role)].reset_index(drop=True)
                    )
                support_table = (
                    pd.concat(cumulative_support, ignore_index=True)
                    if cumulative_support
                    else None
                )
                sampling_started = time.perf_counter()
                graphs, guidance_stats = sample_guided_graphs(
                    feature_store=feature_store,
                    emerg_model=emerg_model,
                    diffusion=diffusion,
                    item_condition=item_condition,
                    gd_config=gd_config,
                    sample_seed=(
                        int(seed) * 1_000_003
                        + int(item_idx) * 97
                        + phase_seed_offset[phase]
                    ),
                    support_table=support_table,
                )
                sampling_seconds = time.perf_counter() - sampling_started
                scores, score_std = predict_graph_ensemble(
                    feature_store,
                    emerg_model,
                    query_rows,
                    graphs,
                )
                phase_predictions = query_rows[
                    ["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]
                ].copy()
                phase_predictions.insert(0, "phase", phase)
                phase_predictions["score"] = scores
                phase_predictions["graph_ensemble_score_std"] = score_std
                phase_predictions["graph_sample_count"] = int(len(graphs))
                predictions.append(phase_predictions)
                diagnostics.append(
                    graph_ensemble_stats(
                        graphs,
                        split,
                        phase,
                        query_rows,
                        guidance_stats,
                        sampling_seconds,
                    )
                )
    finally:
        restore_requires_grad(emerg_model, previous_emerg)
        restore_requires_grad(diffusion, previous_diffusion)
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(diagnostics)


In [12]:
def load_baseline_evaluation_metrics(baseline_manifest: dict[str, Any]) -> pd.DataFrame:
    table = pd.read_csv(baseline_artifact_path(baseline_manifest, "evaluation_metrics"))
    required = {"phase", "f1", "roc_auc", "accuracy", "precision", "recall"}
    if not required.issubset(table.columns):
        raise ValueError("Baseline evaluation_metrics artifact has an unexpected schema")
    return table


def run_emerg_gd_seed(
    baseline_manifest: dict[str, Any],
    seed: int,
) -> dict[str, Any]:
    selected_emerg_config = dict(
        baseline_manifest["selected_config"]["selected_config"]
    )
    gd_config = make_gd_config(seed, selected_emerg_config)
    feature_store = FeatureStore(USERS, ITEMS, TUNING_TRAIN, make_run_config(seed))

    # Tuning stage: train only on tuning_train, then select GD thresholds on validation queries.
    tuning_emerg, tuning_emerg_history, tuning_emerg_seconds = train_emerg_model(
        feature_store,
        TUNING_TRAIN,
        selected_emerg_config,
        seed=seed + 20_001,
        run_name="gd_tuning_emerg_backbone",
    )
    tuning_pseudo = harvest_pseudo_graph_dataset(
        feature_store,
        tuning_emerg,
        TUNING_TRAIN,
        gd_config,
    )
    tuning_diffusion, tuning_diffusion_history, tuning_diffusion_seconds = (
        train_conditional_graph_diffusion(
            tuning_pseudo,
            gd_config,
            seed=seed + 20_002,
            run_name="gd_tuning_diffusion",
        )
    )
    gradient_check = graph_diffusion_gradient_check(
        feature_store,
        tuning_emerg,
        tuning_diffusion,
        TUNING_TRAIN,
    )
    if gradient_check["status"] != "PASS":
        raise RuntimeError("Graph diffusion denoiser is not on the autograd path")
    validation_predictions, validation_diagnostics = score_tasks_with_graph_diffusion(
        feature_store,
        tuning_emerg,
        tuning_diffusion,
        VALIDATION_TASKS,
        gd_config,
        split="validation",
        seed=seed + 20_003,
    )
    validation_thresholds, validation_metrics = summarize_predictions(
        validation_predictions,
        split="validation",
    )
    threshold_by_phase = dict(
        zip(validation_thresholds["phase"], validation_thresholds["threshold"])
    )

    del tuning_emerg, tuning_diffusion
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    # Final stage: use the already-exported final EmerG checkpoint and fit GD only on final_train graphs.
    final_emerg, loaded_config = load_baseline_model(feature_store, baseline_manifest)
    if loaded_config != selected_emerg_config:
        raise ValueError("Selected EmerG config and final checkpoint config disagree")
    final_pseudo = harvest_pseudo_graph_dataset(
        feature_store,
        final_emerg,
        FINAL_TRAIN,
        gd_config,
    )
    final_diffusion, final_diffusion_history, final_diffusion_seconds = (
        train_conditional_graph_diffusion(
            final_pseudo,
            gd_config,
            seed=seed + 30_002,
            run_name="gd_final_diffusion",
        )
    )
    evaluation_predictions, evaluation_diagnostics = score_tasks_with_graph_diffusion(
        feature_store,
        final_emerg,
        final_diffusion,
        EVALUATION_TASKS,
        gd_config,
        split="evaluation",
        seed=seed + 30_003,
    )
    _, evaluation_metrics = summarize_predictions(
        evaluation_predictions,
        split="evaluation",
        thresholds=threshold_by_phase,
    )
    baseline_metrics = load_baseline_evaluation_metrics(baseline_manifest)
    paired_metrics = baseline_metrics.merge(
        evaluation_metrics,
        on="phase",
        suffixes=("_emerg", "_emerg_gd"),
        validate="one_to_one",
    )
    for metric in ["f1", "roc_auc", "accuracy", "precision", "recall"]:
        paired_metrics[f"{metric}_delta"] = (
            paired_metrics[f"{metric}_emerg_gd"] - paired_metrics[f"{metric}_emerg"]
        )

    diffusion_history = pd.concat(
        [tuning_diffusion_history, final_diffusion_history], ignore_index=True
    )
    pseudo_graph_stats = pd.concat(
        [
            tuning_pseudo["stats"].assign(stage="tuning"),
            final_pseudo["stats"].assign(stage="final"),
        ],
        ignore_index=True,
    )
    graph_diagnostics = pd.concat(
        [validation_diagnostics, evaluation_diagnostics], ignore_index=True
    )
    timing = pd.DataFrame(
        [
            {
                "stage": "tuning_emerg_backbone",
                "seconds": float(tuning_emerg_seconds),
            },
            {
                "stage": "tuning_diffusion",
                "seconds": float(tuning_diffusion_seconds),
            },
            {
                "stage": "final_diffusion",
                "seconds": float(final_diffusion_seconds),
            },
        ]
    )

    expected_validation_predictions = int(
        VALIDATION_TASKS["role"].eq("query").sum() * len(PHASES)
    )
    expected_evaluation_predictions = int(
        EVALUATION_TASKS["role"].eq("query").sum() * len(PHASES)
    )
    checks: list[dict[str, Any]] = []
    pre_export_check(
        checks,
        "diffusion gradient flow",
        gradient_check["status"] == "PASS",
        gradient_check,
        "PASS with positive finite gradient norm",
    )
    pre_export_check(
        checks,
        "validation predictions complete",
        len(validation_predictions) == expected_validation_predictions,
        len(validation_predictions),
        expected_validation_predictions,
    )
    pre_export_check(
        checks,
        "evaluation predictions complete",
        len(evaluation_predictions) == expected_evaluation_predictions,
        len(evaluation_predictions),
        expected_evaluation_predictions,
    )
    pre_export_check(
        checks,
        "phase thresholds complete",
        set(threshold_by_phase) == set(PHASES),
        sorted(threshold_by_phase),
        sorted(PHASES),
    )
    pre_export_check(
        checks,
        "final train excludes evaluation items",
        set(FINAL_TRAIN["item_idx"]).isdisjoint(set(EVALUATION_TASKS["item_idx"])),
        True,
        True,
    )
    pre_export_check(
        checks,
        "all numerical outputs finite",
        numeric_tables_are_finite(
            tuning_emerg_history,
            diffusion_history,
            validation_thresholds,
            validation_metrics,
            validation_predictions,
            evaluation_metrics,
            evaluation_predictions,
            graph_diagnostics,
            paired_metrics,
            timing,
        ),
        True,
        True,
    )
    failed = [row["check"] for row in checks if row["status"] != "PASS"]
    if failed:
        raise RuntimeError(f"EmerG+GD checks failed for seed {seed}: {', '.join(failed)}")

    return {
        "seed": int(seed),
        "feature_store": feature_store,
        "final_emerg": final_emerg,
        "final_diffusion": final_diffusion,
        "gd_config": gd_config,
        "selected_emerg_config": selected_emerg_config,
        "baseline_manifest": baseline_manifest,
        "threshold_by_phase": threshold_by_phase,
        "gradient_check": gradient_check,
        "checks": checks,
        "output_tables": {
            "tuning_emerg_history": tuning_emerg_history,
            "diffusion_history": diffusion_history,
            "pseudo_graph_stats": pseudo_graph_stats,
            "validation_thresholds": validation_thresholds,
            "validation_metrics": validation_metrics,
            "validation_predictions": validation_predictions,
            "evaluation_metrics": evaluation_metrics,
            "evaluation_predictions": evaluation_predictions,
            "graph_diagnostics": graph_diagnostics,
            "paired_seed_metrics": paired_metrics,
            "timing": timing,
        },
    }


def verify_gd_generation(manifest: dict[str, Any], expected_seed: int) -> dict[str, Any]:
    verification = {"manifest_verified": False, "artifact_hashes_verified": False, "error": ""}
    try:
        generation_root = GD_MODEL_OUTPUT_ROOT / "generations" / manifest["bundle_id"]
        manifest_path = resolve_inside(ARTIFACT_ROOT, manifest["bundle_manifest"])
        loaded = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest_ok = (
            loaded == json_ready(manifest)
            and loaded.get("model_status") == "PASS"
            and loaded.get("model_name") == "EmerG+GD"
            and int(loaded.get("run_config", {}).get("seed", -1)) == int(expected_seed)
            and manifest_path.resolve() == (generation_root / "manifest.json").resolve()
        )
        hashes_ok = True
        paths_ok = True
        for artifact in loaded.get("artifacts", {}).values():
            artifact_path = resolve_inside(ARTIFACT_ROOT, artifact["path"])
            try:
                artifact_path.resolve().relative_to(generation_root.resolve())
            except ValueError:
                paths_ok = False
            if not artifact_path.is_file() or sha256_file(artifact_path) != artifact["sha256"]:
                hashes_ok = False
        verification["manifest_verified"] = bool(manifest_ok and paths_ok)
        verification["artifact_hashes_verified"] = bool(hashes_ok)
    except Exception as error:
        verification["error"] = f"{type(error).__name__}: {error}"
    verification["verified"] = bool(
        verification["manifest_verified"] and verification["artifact_hashes_verified"]
    )
    return verification


def export_emerg_gd_seed(run_payload: dict[str, Any], bundle_id: str) -> dict[str, Any]:
    seed = int(run_payload["seed"])
    generation_root = GD_MODEL_OUTPUT_ROOT / "generations" / bundle_id
    staging_root = GD_MODEL_OUTPUT_ROOT / f".staging-{bundle_id}"
    pointer_path = GD_MODEL_OUTPUT_ROOT / "manifest.json"
    staging_root.mkdir(parents=True, exist_ok=False)
    artifacts: dict[str, Any] = {}
    try:
        for name, table in run_payload["output_tables"].items():
            staging_path = staging_root / f"{name}.csv"
            write_csv(staging_path, table)
            artifacts[name] = {
                "path": relative_output(generation_root / f"{name}.csv"),
                "sha256": sha256_file(staging_path),
                "rows": len(table),
            }

        checkpoint_path = staging_root / "graph_diffusion.pt"
        torch.save(
            {
                "schema_version": run_payload["gd_config"]["schema_version"],
                "diffusion_state_dict": run_payload["final_diffusion"].state_dict(),
                "gd_config": run_payload["gd_config"],
                "selected_emerg_config": run_payload["selected_emerg_config"],
                "baseline_bundle_id": run_payload["baseline_manifest"]["bundle_id"],
                "baseline_checkpoint_sha256": run_payload["baseline_manifest"]["artifacts"]["final_model_checkpoint"]["sha256"],
                "threshold_by_phase": run_payload["threshold_by_phase"],
                "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            checkpoint_path,
        )
        artifacts["graph_diffusion_checkpoint"] = {
            "path": relative_output(generation_root / "graph_diffusion.pt"),
            "sha256": sha256_file(checkpoint_path),
        }

        evaluation_metrics = run_payload["output_tables"]["evaluation_metrics"]
        paired_metrics = run_payload["output_tables"]["paired_seed_metrics"]
        manifest = {
            "model_schema_version": run_payload["gd_config"]["schema_version"],
            "model_status": "PASS",
            "model_name": "EmerG+GD",
            "bundle_id": bundle_id,
            "bundle_manifest": relative_output(generation_root / "manifest.json"),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "upstream_protocol": {
                "schema_version": PROTOCOL_MANIFEST["protocol_schema_version"],
                "bundle_id": PROTOCOL_MANIFEST["bundle_id"],
                "pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            "baseline_emerg": {
                "bundle_id": run_payload["baseline_manifest"]["bundle_id"],
                "checkpoint_sha256": run_payload["baseline_manifest"]["artifacts"]["final_model_checkpoint"]["sha256"],
            },
            "run_config": run_payload["gd_config"],
            "threshold_by_phase": run_payload["threshold_by_phase"],
            "gradient_check": run_payload["gradient_check"],
            "checks": run_payload["checks"],
            "summary": {
                "mean_evaluation_f1": float(evaluation_metrics["f1"].mean()),
                "mean_evaluation_auc": float(evaluation_metrics["roc_auc"].mean()),
                "mean_paired_f1_delta": float(paired_metrics["f1_delta"].mean()),
                "evaluation_prediction_rows": int(len(run_payload["output_tables"]["evaluation_predictions"])),
            },
            "artifacts": artifacts,
            "output_schemas": {
                name: csv_schema(table)
                for name, table in run_payload["output_tables"].items()
            },
            "phase_contract": PROTOCOL_MANIFEST["phase_contract"],
            "training_contract": {
                "pseudo_targets": "converged EmerG continuous graphs harvested only from the active training split",
                "conditioning": GD_CONDITION_FIELDS,
                "sampling": "short deterministic DDIM-style chain with independent Gaussian initial states",
                "warm_guidance": run_payload["gd_config"]["warm_support_policy"],
                "thresholds": "selected on validation query rows only and frozen for evaluation",
            },
        }
        manifest_text = json.dumps(json_ready(manifest), indent=2, sort_keys=True, allow_nan=False) + "\n"
        write_text(staging_root / "manifest.json", manifest_text)
        generation_root.parent.mkdir(parents=True, exist_ok=True)
        staging_root.replace(generation_root)
        verification = verify_gd_generation(manifest, seed)
        if not verification["verified"]:
            raise RuntimeError(verification["error"] or str(verification))
        write_text(pointer_path, manifest_text)
    except Exception:
        if staging_root.exists():
            shutil.rmtree(staging_root, ignore_errors=True)
        if generation_root.exists():
            shutil.rmtree(generation_root, ignore_errors=True)
        raise
    return {
        "manifest": manifest,
        "verification": verification,
        "generation_manifest": generation_root / "manifest.json",
        "pointer": pointer_path,
    }


In [ ]:
# Execute the paired multi-seed EmerG+GD experiment and build the requested scenario table.
if not EMERG_PASS:
    raise RuntimeError("Inherited EmerG stage must be READY before EmerG+GD can run")

BASELINE_EXPORT_BY_SEED = {
    int(completed["seed"]): completed for completed in COMPLETED_EXPORTS
}
GD_RUN_REGISTRY_ROWS: list[dict[str, Any]] = []
GD_COMPLETED_EXPORTS: list[dict[str, Any]] = []
GD_PAIRED_METRIC_PARTS: list[pd.DataFrame] = []

for target_seed in TARGET_SEEDS:
    seed_started = time.perf_counter()
    planned_bundle_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        + f"-s{target_seed}-"
        + uuid.uuid4().hex[:12]
    )
    registry_row = {
        "seed": int(target_seed),
        "status": "BLOCKED",
        "baseline_bundle_id": "",
        "gd_bundle_id": "",
        "mean_emerg_f1": np.nan,
        "mean_emerg_gd_f1": np.nan,
        "mean_f1_delta": np.nan,
        "manifest_verified": False,
        "artifact_hashes_verified": False,
        "error": "",
    }
    run_payload = None
    try:
        baseline_export = BASELINE_EXPORT_BY_SEED[int(target_seed)]
        baseline_manifest = baseline_export["manifest"]
        registry_row["baseline_bundle_id"] = baseline_manifest["bundle_id"]
        run_payload = run_emerg_gd_seed(
            baseline_manifest=baseline_manifest,
            seed=int(target_seed),
        )
        export = export_emerg_gd_seed(run_payload, planned_bundle_id)
        paired = run_payload["output_tables"]["paired_seed_metrics"].copy()
        paired.insert(0, "seed", int(target_seed))
        GD_PAIRED_METRIC_PARTS.append(paired)
        baseline_mean = float(paired["f1_emerg"].mean())
        gd_mean = float(paired["f1_emerg_gd"].mean())
        registry_row.update(
            {
                "status": "READY",
                "gd_bundle_id": export["manifest"]["bundle_id"],
                "mean_emerg_f1": baseline_mean,
                "mean_emerg_gd_f1": gd_mean,
                "mean_f1_delta": gd_mean - baseline_mean,
                "manifest_verified": export["verification"]["manifest_verified"],
                "artifact_hashes_verified": export["verification"]["artifact_hashes_verified"],
            }
        )
        GD_COMPLETED_EXPORTS.append(
            {
                "seed": int(target_seed),
                "manifest": export["manifest"],
                "verification": export["verification"],
            }
        )
    except Exception as error:
        registry_row["error"] = f"{type(error).__name__}: {error}"
    finally:
        registry_row["elapsed_seconds"] = time.perf_counter() - seed_started
        GD_RUN_REGISTRY_ROWS.append(registry_row)
        if run_payload is not None:
            run_payload["final_emerg"] = None
            run_payload["final_diffusion"] = None
            run_payload["feature_store"] = None
        run_payload = None
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

GD_RUN_REGISTRY = pd.DataFrame(GD_RUN_REGISTRY_ROWS)
display(GD_RUN_REGISTRY)

GD_PASS = (
    len(GD_COMPLETED_EXPORTS) == len(TARGET_SEEDS)
    and all(row["status"] == "READY" for row in GD_RUN_REGISTRY_ROWS)
    and all(
        row["manifest_verified"] and row["artifact_hashes_verified"]
        for row in GD_RUN_REGISTRY_ROWS
    )
)
if not GD_PASS:
    raise RuntimeError("EmerG+GD multi-seed run is BLOCKED; inspect GD_RUN_REGISTRY")

PAIRED_SEED_SCENARIO_RESULTS = pd.concat(GD_PAIRED_METRIC_PARTS, ignore_index=True)
summary_rows: list[dict[str, Any]] = []
for scenario, rows in PAIRED_SEED_SCENARIO_RESULTS.groupby("phase", sort=False):
    emerg_f1 = rows["f1_emerg"].to_numpy(dtype=np.float64)
    gd_f1 = rows["f1_emerg_gd"].to_numpy(dtype=np.float64)
    emerg_auc = rows["roc_auc_emerg"].to_numpy(dtype=np.float64)
    gd_auc = rows["roc_auc_emerg_gd"].to_numpy(dtype=np.float64)
    summary_rows.append(
        {
            "scenario": scenario,
            "seeds": len(rows),
            "emerg_f1_mean": float(emerg_f1.mean()),
            "emerg_f1_std": float(emerg_f1.std(ddof=1)) if len(rows) > 1 else 0.0,
            "emerg_gd_f1_mean": float(gd_f1.mean()),
            "emerg_gd_f1_std": float(gd_f1.std(ddof=1)) if len(rows) > 1 else 0.0,
            "f1_delta_mean": float((gd_f1 - emerg_f1).mean()),
            "f1_relative_change_pct": float(
                100.0 * (gd_f1.mean() - emerg_f1.mean()) / max(abs(emerg_f1.mean()), 1e-12)
            ),
            "emerg_auc_mean": float(emerg_auc.mean()),
            "emerg_gd_auc_mean": float(gd_auc.mean()),
            "auc_delta_mean": float((gd_auc - emerg_auc).mean()),
        }
    )

overall_rows = PAIRED_SEED_SCENARIO_RESULTS
summary_rows.append(
    {
        "scenario": "Overall mean across scenarios",
        "seeds": int(overall_rows["seed"].nunique()),
        "emerg_f1_mean": float(overall_rows["f1_emerg"].mean()),
        "emerg_f1_std": float(overall_rows["f1_emerg"].std(ddof=1)),
        "emerg_gd_f1_mean": float(overall_rows["f1_emerg_gd"].mean()),
        "emerg_gd_f1_std": float(overall_rows["f1_emerg_gd"].std(ddof=1)),
        "f1_delta_mean": float(overall_rows["f1_delta"].mean()),
        "f1_relative_change_pct": float(
            100.0
            * overall_rows["f1_delta"].mean()
            / max(abs(overall_rows["f1_emerg"].mean()), 1e-12)
        ),
        "emerg_auc_mean": float(overall_rows["roc_auc_emerg"].mean()),
        "emerg_gd_auc_mean": float(overall_rows["roc_auc_emerg_gd"].mean()),
        "auc_delta_mean": float(overall_rows["roc_auc_delta"].mean()),
    }
)

EMERG_VS_EMERG_GD_COMPARISON = pd.DataFrame(summary_rows)
comparison_root = GD_MODEL_OUTPUT_ROOT / "comparisons"
comparison_root.mkdir(parents=True, exist_ok=True)
write_csv(
    comparison_root / "emerg_vs_emerg_gd_by_seed_and_scenario.csv",
    PAIRED_SEED_SCENARIO_RESULTS,
)
write_csv(
    comparison_root / "emerg_vs_emerg_gd_summary.csv",
    EMERG_VS_EMERG_GD_COMPARISON,
)

comparison_display = EMERG_VS_EMERG_GD_COMPARISON.copy()
for column in comparison_display.columns:
    if column not in {"scenario", "seeds"}:
        comparison_display[column] = comparison_display[column].map(lambda value: f"{value:.6f}")

display(Markdown("## Final paired comparison: EmerG vs EmerG + Graph Diffusion"))
display(comparison_display)
display(
    Markdown(
        "Positive `f1_delta_mean` and `auc_delta_mean` favor EmerG+GD. "
        "The Cold row is the primary test of the distributional prior; Warm A/B/C also test guided adaptation."
    )
)
display(Markdown("### Notebook 051 EmerG+GD: READY"))


## Output contract

After a successful full run, this notebook produces:

- the inherited verified EmerG bundles for all target seeds;
- one verified EmerG+GD bundle per seed under `artifacts/models/ml-1m/emerg-gd-v1`;
- graph-diffusion checkpoints, pseudo-graph diagnostics, uncertainty-aware predictions,
  validation thresholds, evaluation metrics, and paired per-seed metrics;
- `emerg_vs_emerg_gd_by_seed_and_scenario.csv`;
- `emerg_vs_emerg_gd_summary.csv`, displayed above as the final comparison table.

A positive delta is not forced or assumed: the table reports the observed paired result.
